# 09 — Error and Event Analysis

**Purpose:** Explain **why** the frozen NB08 aggregate metrics look the way they do — moving from point/sequence-level metrics to false-positive patterns, false-negative patterns, attack-event behaviour, event coverage, detection delay, and TCN-vs-LSTM complementarity.

**No model is trained, tuned, retrained, threshold-calibrated, or altered in this notebook.** All scores, thresholds, and sequence indices are loaded read-only from `nb08_artifacts\`. No write touches `nb08_artifacts\`.

## Prerequisite gates
- **GAP 1 (`List_of_attacks_Final.xlsx`)** must be present for Phases 5 and 6 (event-level) to run. If absent, Cells 5–6 set `HAVE_SCENARIOS=False` and Phases 5–6 event-level cells print BLOCKED and skip. **Phases 2–4 and point-level Phase 6 still run.**
- **GAP 2 (derived normal-partition timestamps)** is resolved by Cell 4. This must succeed before any temporal analysis.

## How to read this notebook
- **Phase 2** reconstructs TP/FP/TN/FN at the frozen thresholds and **asserts** equality with `evaluation_summary.csv`. If this fails, the notebook stops — nothing downstream is valid.
- **Phase 3 / Phase 4** characterise FPs and FNs at sequence level (clustering by 300 s gap, per-sensor attribution).
- **Phase 5** (gated) computes per-scenario detection, coverage, and delay for all 36 SUTD scenarios.
- **Phase 6** tests whether TCN-AE and LSTM-AE errors are **complementary**. Cell 26 is **descriptive only** — it does **not** select an ensemble.

In [1]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

In [2]:

from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)


# Project paths
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (path / "nb08_artifacts").is_dir()
        and (path / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected both 'nb08_artifacts' and 'data/raw'."
    )

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
NB08_DIR = PROJECT_ROOT / "nb08_artifacts"
NB09_OUT_DIR = PROJECT_ROOT / "nb09_artifacts"

NB09_OUT_DIR.mkdir(parents=True, exist_ok=True)


# Source data
ATTACK_LIST_PATH = DATA_RAW_DIR / "List_of_attacks_Final.xlsx"


def find_source_file(filename):
    candidates = (
        DATA_RAW_DIR / filename,
        PROJECT_ROOT / filename,
    )

    for path in candidates:
        if path.is_file():
            return path

    raise FileNotFoundError(
        f"Required source file '{filename}' was not found."
    )


NORMAL_CSV = find_source_file("normal.csv")
ATTACK_CSV = find_source_file("attack.csv")


# Frozen Notebook 08 artifacts
FINAL_SEQUENCE_SCORES = NB08_DIR / "final_sequence_scores.npz"
FINAL_OBSERVATION_SCORES = NB08_DIR / "final_observation_scores.npz"
FROZEN_THRESHOLDS = NB08_DIR / "frozen_thresholds.json"
EVALUATION_SUMMARY = NB08_DIR / "evaluation_summary.csv"
EVALUATION_SEQUENCES = NB08_DIR / "evaluation_sequences.npz"
ATTACK_EVALUATION_SEQUENCES = NB08_DIR / "attack_evaluation_sequences.npz"

LSTM_ATTACK_SCORES = NB08_DIR / "lstm_attack_scores.npz"
TCN_ATTACK_SCORES = NB08_DIR / "tcn_attack_scores.npz"
USAD_ATTACK_SCORES = NB08_DIR / "usad_attack_scores.npz"


# Validate required inputs
REQUIRED_FILES = [
    ATTACK_LIST_PATH,
    NORMAL_CSV,
    ATTACK_CSV,
    FINAL_SEQUENCE_SCORES,
    FINAL_OBSERVATION_SCORES,
    FROZEN_THRESHOLDS,
    EVALUATION_SUMMARY,
    EVALUATION_SEQUENCES,
    ATTACK_EVALUATION_SEQUENCES,
    LSTM_ATTACK_SCORES,
    TCN_ATTACK_SCORES,
    USAD_ATTACK_SCORES,
]

MISSING_FILES = [
    path for path in REQUIRED_FILES
    if not path.is_file()
]

if MISSING_FILES:
    missing_files = "\n".join(
        f"- {path.name}" for path in MISSING_FILES
    )
    raise FileNotFoundError(
        f"Required Notebook 09 inputs are missing:\n{missing_files}"
    )


# Methods evaluated in Notebook 09
METHODS = [
    "rule",
    "rolling",
    "simple",
    "if",
    "ocsvm",
    "lof",
    "lstm",
    "tcn",
    "usad",
]

OBS_METHODS = [
    "rule",
    "rolling",
    "simple",
    "if",
    "ocsvm",
    "lof",
]


In [3]:
import json
import joblib

# Purpose: Load frozen NB08 artifacts and scaler feature names; assert shape invariants.
seq_npz  = np.load(NB08_DIR / "final_sequence_scores.npz")
obs_npz  = np.load(NB08_DIR / "final_observation_scores.npz")
eval_npz = np.load(NB08_DIR / "evaluation_sequences.npz")
atk_eval = np.load(NB08_DIR / "attack_evaluation_sequences.npz")

SEQ_SCORES = {k: seq_npz[k] for k in seq_npz.files}
OBS_SCORES = {k: obs_npz[k] for k in obs_npz.files}

with open(NB08_DIR / "frozen_thresholds.json") as f:
    THRESHOLDS = json.load(f)["thresholds"]

X_threshold_seq        = eval_npz["X_threshold_seq"]
X_validation_seq       = eval_npz["X_validation_seq"]
X_attack_seq           = atk_eval["X_attack_seq"]
threshold_end_indices  = eval_npz["threshold_end_indices"]
validation_end_indices = eval_npz["validation_end_indices"]
attack_end_indices     = atk_eval["attack_end_indices"]

lstm_atk = np.load(NB08_DIR / "lstm_attack_scores.npz")
tcn_atk  = np.load(NB08_DIR / "tcn_attack_scores.npz")
usad_atk = np.load(NB08_DIR / "usad_attack_scores.npz")

ATK_TIMESTAMPS = tcn_atk["attack_timestamps"]
ATK_LABELS     = tcn_atk["attack_labels"]

assert np.array_equal(ATK_TIMESTAMPS, lstm_atk["attack_timestamps"]), "LSTM/TCN attack timestamps differ"
assert np.array_equal(ATK_TIMESTAMPS, usad_atk["attack_timestamps"]), "USAD/TCN attack timestamps differ"
assert np.array_equal(ATK_LABELS, lstm_atk["attack_labels"]), "LSTM/TCN attack labels differ"
assert np.array_equal(ATK_LABELS, usad_atk["attack_labels"]), "USAD/TCN attack labels differ"

scaler = joblib.load(NB08_DIR / "scaler.joblib")
FEATURE_NAMES = list(scaler.feature_names_in_)

assert len(SEQ_SCORES) == 30, f"Expected 30 sequence artifact keys, got {len(SEQ_SCORES)}"
assert len(OBS_SCORES) == 18, f"Expected 18 observation-score keys, got {len(OBS_SCORES)}"

assert set(SEQ_SCORES) == (
    {f"{method}_{split}_seq_scores"
     for method in METHODS
     for split in ("threshold", "validation", "attack")}
    | {"threshold_end_indices", "validation_end_indices", "attack_end_indices"}
), "Sequence artifact keys mismatch"

assert set(OBS_SCORES) == {
    f"{method}_{split}_scores"
    for method in ("rule", "rolling", "simple", "if", "ocsvm", "lof")
    for split in ("threshold", "validation", "attack")
}, "Observation artifact keys mismatch"

assert len(THRESHOLDS) == 9 and set(THRESHOLDS) == set(METHODS), "THRESHOLDS keys mismatch"
assert len(FEATURE_NAMES) == 42, f"FEATURE_NAMES expected 42, got {len(FEATURE_NAMES)}"

assert X_threshold_seq.shape == (47408, 30, 42), f"Unexpected threshold sequence shape: {X_threshold_seq.shape}"
assert X_validation_seq.shape == (79031, 30, 42), f"Unexpected validation sequence shape: {X_validation_seq.shape}"
assert X_attack_seq.shape == (54592, 30, 42), f"Unexpected attack sequence shape: {X_attack_seq.shape}"

assert len(threshold_end_indices) == len(X_threshold_seq)
assert len(validation_end_indices) == len(X_validation_seq)
assert len(attack_end_indices) == len(X_attack_seq)

for name, scores in SEQ_SCORES.items():
    assert scores.ndim == 1 and np.isfinite(scores).all(), f"Invalid sequence scores: {name}"

for name, scores in OBS_SCORES.items():
    assert scores.ndim == 1 and np.isfinite(scores).all(), f"Invalid observation scores: {name}"

assert all(np.all(np.diff(indices) > 0) for indices in (
    threshold_end_indices, validation_end_indices, attack_end_indices
)), "Sequence end indices must be strictly increasing"

assert len(ATK_TIMESTAMPS) == 54592 and len(ATK_LABELS) == 54592
assert np.isfinite(X_threshold_seq).all()
assert np.isfinite(X_validation_seq).all()
assert np.isfinite(X_attack_seq).all()

print(f"SEQ_SCORES={len(SEQ_SCORES)}  OBS_SCORES={len(OBS_SCORES)}  THRESHOLDS={len(THRESHOLDS)}  FEATURES={len(FEATURE_NAMES)}")
print(f"end_indices: thr={len(threshold_end_indices)}  val={len(validation_end_indices)}  atk={len(attack_end_indices)}")
print("Artifact loading and structural validation passed.")

SEQ_SCORES=30  OBS_SCORES=18  THRESHOLDS=9  FEATURES=42
end_indices: thr=47408  val=79031  atk=54592
Artifact loading and structural validation passed.


In [4]:
# Purpose: Inspect the CSV header to identify why Timestamp is missing.

import csv

print("File selected:", NORMAL_CSV.name)
print("File exists:", NORMAL_CSV.is_file())
print("File size (MB):", round(NORMAL_CSV.stat().st_size / (1024 ** 2), 2))

with open(NORMAL_CSV, "r", encoding="utf-8-sig", newline="") as file:
    first_line = file.readline()

print("\nRaw header:")
print(repr(first_line[:500]))

for delimiter in [",", ";", "\t", "|"]:
    columns = next(csv.reader([first_line], delimiter=delimiter))
    cleaned = [column.strip() for column in columns]

    print(f"\nDelimiter: {repr(delimiter)}")
    print("Column count:", len(columns))
    print("First five columns:", repr(cleaned[:5]))
    print("Contains Timestamp:", "Timestamp" in cleaned)

File selected: normal.csv
File exists: True
File size (MB): 383.73

Raw header:
' Timestamp,FIT101,LIT101, MV101,P101,P102, AIT201,AIT202,AIT203,FIT201, MV201, P201, P202,P203, P204,P205,P206,DPIT301,FIT301,LIT301,MV301,MV302, MV303,MV304,P301,P302,AIT401,AIT402,FIT401,LIT401,P401,P402,P403,P404,UV401,AIT501,AIT502,AIT503,AIT504,FIT501,FIT502,FIT503,FIT504,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack\n'

Delimiter: ','
Column count: 53
First five columns: ['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101']
Contains Timestamp: True

Delimiter: ';'
Column count: 1
First five columns: ['Timestamp,FIT101,LIT101, MV101,P101,P102, AIT201,AIT202,AIT203,FIT201, MV201, P201, P202,P203, P204,P205,P206,DPIT301,FIT301,LIT301,MV301,MV302, MV303,MV304,P301,P302,AIT401,AIT402,FIT401,LIT401,P401,P402,P403,P404,UV401,AIT501,AIT502,AIT503,AIT504,FIT501,FIT502,FIT503,FIT504,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack']
Contains Timestamp: False

Delimiter: '\t'

In [5]:
# Purpose: Resolve GAP 2 — derive threshold/validation partition timestamps
# from normal.csv using the NB06 eligibility filter; save to nb09_artifacts.

normal_df = pd.read_csv(NORMAL_CSV, skipinitialspace=True)
normal_df.columns = normal_df.columns.str.strip()

assert "Timestamp" in normal_df.columns, (
    f"Timestamp column not found. First columns: {normal_df.columns[:5].tolist()}"
)

normal_df["Timestamp"] = pd.to_datetime(
    normal_df["Timestamp"],
    dayfirst=True,
    errors="coerce"
)

SENSOR_COLS = [
    c for c in normal_df.columns
    if c not in ("Timestamp", "Normal/Attack")
]
assert len(SENSOR_COLS) == 51, (
    f"Expected 51 sensors, got {len(SENSOR_COLS)}"
)

mask = (
    normal_df["Timestamp"].notna()
    & normal_df[SENSOR_COLS].notna().all(axis=1)
)
eligible_normal = normal_df.loc[mask].reset_index(drop=True)

assert len(eligible_normal) == 395298, (
    f"eligible_normal expected 395298, got {len(eligible_normal)}"
)

TRAIN_N, MONITOR_N, THR_N, VAL_N = 221366, 47435, 47437, 79060
assert TRAIN_N + MONITOR_N + THR_N + VAL_N == len(eligible_normal)

thr_start = TRAIN_N + MONITOR_N
val_start = thr_start + THR_N

thr_ts_raw = (
    eligible_normal.iloc[thr_start:thr_start + THR_N]["Timestamp"]
    .reset_index(drop=True)
)
val_ts_raw = (
    eligible_normal.iloc[val_start:val_start + VAL_N]["Timestamp"]
    .reset_index(drop=True)
)

threshold_ts = (
    thr_ts_raw.iloc[threshold_end_indices]
    .reset_index(drop=True)
)
validation_ts = (
    val_ts_raw.iloc[validation_end_indices]
    .reset_index(drop=True)
)

assert len(threshold_ts) == 47408
assert len(validation_ts) == 79031

assert (threshold_ts.diff().dropna() > pd.Timedelta(0)).all(), (
    "Threshold timestamps are not strictly increasing"
)
assert (validation_ts.diff().dropna() > pd.Timedelta(0)).all(), (
    "Validation timestamps are not strictly increasing"
)

assert str(threshold_ts.iloc[-1])[:19] == "2016-01-01 15:51:19", (
    f"Unexpected threshold end timestamp: {threshold_ts.iloc[-1]}"
)
assert threshold_ts.iloc[-1] < validation_ts.iloc[0]

np.savez(
    NB09_OUT_DIR / "derived_timestamps.npz",
    threshold_ts=threshold_ts.values,
    validation_ts=validation_ts.values,
)

print(
    f"threshold : {threshold_ts.iloc[0]} -> "
    f"{threshold_ts.iloc[-1]} ({len(threshold_ts)})"
)
print(
    f"validation: {validation_ts.iloc[0]} -> "
    f"{validation_ts.iloc[-1]} ({len(validation_ts)})"
)
print("Timestamp derivation and validation passed.")


threshold : 2016-01-01 02:23:47 -> 2016-01-01 15:51:19 (47408)
validation: 2016-01-01 15:51:49 -> 2016-01-02 14:59:59 (79031)
Timestamp derivation and validation passed.


In [6]:
# Purpose: Inspect the scenario workbook before normalizing its schema.

assert ATTACK_LIST_PATH.is_file(), (
    f"Scenario workbook not found: {ATTACK_LIST_PATH.name}"
)

xls = pd.ExcelFile(ATTACK_LIST_PATH)
print("Sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    raw = pd.read_excel(ATTACK_LIST_PATH, sheet_name=sheet)

    print(f"\nSheet: {sheet}")
    print("Shape:", raw.shape)
    print("Columns:", [repr(c) for c in raw.columns])
    print("\nFirst five rows:")
    display(raw.head())

Sheets: ['Sheet1']

Sheet: Sheet1
Shape: (49, 9)
Columns: ["'Attack #'", "'Start Time'", "'End Time'", "'Attack Point'", "'Start State'", "'Attack'", "'Actual Change'", "'Expected Impact or attacker intent'", "'Unexpected Outcome'"]

First five rows:


,Attack #,Start Time,End Time,Attack Point,Start State,Attack,Actual Change,Expected Impact or attacker intent,Unexpected Outcome
0,1,2015-12-28 10:29:14,10:44:53,MV-101,MV-101 is closed,Open MV-101,Yes,Tank overflow,NaN
1,2,2015-12-28 10:51:08,10:58:30,P-102,P-101 is on where as P-102 is off,Turn on P-102,Yes,Pipe bursts,NaN
2,3,2015-12-28 11:22:00,11:28:22,LIT-101,Water level between L and H,Increase by 1 mm every second,No,Tank Underflow; Damage P-101,NaN
3,4,2015-12-28 11:47:39,11:54:08,MV-504,MV-504 is closed,Open MV-504,Yes,Halt RO shut down sequence; Reduce life of RO,No impact
4,5,2015-12-28 11:58:20,NaN,No Physical Impact Attack,NaN,NaN,NaN,NaN,NaN


In [7]:
# Purpose: Inspect original Start Time and End Time values and types.

raw = pd.read_excel(ATTACK_LIST_PATH, sheet_name="Sheet1")
raw.columns = raw.columns.astype(str).str.strip()

for column in ["Attack #", "Start Time", "End Time"]:
    print(f"\n{column}")
    print("Data types:", raw[column].map(lambda value: type(value).__name__).value_counts().to_dict())
    display(raw[["Attack #", column]].head(12))

print("\nRows with missing end times:")
display(
    raw.loc[
        raw["End Time"].isna(),
        ["Attack #", "Start Time", "End Time", "Attack Point", "Attack"]
    ].head(15)
)


Attack #
Data types: {'int': 41, 'float': 7, 'str': 1}


,Attack #,Attack #
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5
5,6,6
6,7,7
7,8,8
8,9,9
9,10,10



Start Time
Data types: {'datetime': 41, 'float': 4, 'str': 4}


,Attack #,Start Time
0,1,2015-12-28 10:29:14
1,2,2015-12-28 10:51:08
2,3,2015-12-28 11:22:00
3,4,2015-12-28 11:47:39
4,5,2015-12-28 11:58:20
5,6,2015-12-28 12:00:55
6,7,2015-12-28 12:08:25
7,8,2015-12-28 13:10:10
8,9,2015-12-28 14:15:00
9,10,2015-12-28 14:16:20



End Time
Data types: {'time': 36, 'float': 13}


,Attack #,End Time
0,1,10:44:53
1,2,10:58:30
2,3,11:28:22
3,4,11:54:08
4,5,NaN
5,6,12:04:10
6,7,12:15:33
7,8,13:26:13
8,9,NaN
9,10,14:19:00



Rows with missing end times:


,Attack #,Start Time,End Time,Attack Point,Attack
4,5,2015-12-28 11:58:20,NaN,No Physical Impact Attack,NaN
8,9,2015-12-28 14:15:00,NaN,No Physical Impact Attack,NaN
11,12,2015-12-29 11:10:40,NaN,No Physical Impact Attack,NaN
14,15,2015-12-29 11:52:01,NaN,No Physical Impact Attack,NaN
17,18,2015-12-29 18:08:55,NaN,No Physical Impact Attack,NaN
41,NaN,NaN,NaN,NaN,NaN
42,NaN,NaN,NaN,NaN,NaN
43,NaN,Single Stage Single Point Attacks,NaN,NaN,NaN
44,NaN,Single Stage Multi Point Attacks,NaN,NaN,NaN
45,NaN,Multi Stage Single Point Attacks,NaN,NaN,NaN


In [8]:

# Purpose: Normalize the SUTD scenario workbook using its observed schema.
# Preserve incomplete numbered records and separately identify non-data rows.

from datetime import datetime, date, time, timedelta

HAVE_SCENARIOS = False
attack_scenarios = None

if not ATTACK_LIST_PATH.is_file():
    print(f"BLOCKED: {ATTACK_LIST_PATH.name} was not found.")
else:
    raw = pd.read_excel(ATTACK_LIST_PATH, sheet_name="Sheet1")
    raw.columns = raw.columns.astype(str).str.strip()

    required_columns = [
        "Attack #", "Start Time", "End Time", "Attack Point",
        "Start State", "Attack", "Actual Change",
        "Expected Impact or attacker intent", "Unexpected Outcome",
    ]
    missing_columns = [c for c in required_columns if c not in raw.columns]
    if missing_columns:
        raise ValueError(f"Missing workbook columns: {missing_columns}")

    def parse_start(value):
        if pd.isna(value) or not isinstance(value, (datetime, date, pd.Timestamp)):
            return pd.NaT
        return pd.Timestamp(value)

    def parse_end(value, start):
        if pd.isna(value) or pd.isna(start):
            return pd.NaT

        # Excel time-only cells are returned by pandas as datetime.time.
        if isinstance(value, time):
            end = pd.Timestamp(start.date()) + pd.Timedelta(
                hours=value.hour,
                minutes=value.minute,
                seconds=value.second,
                microseconds=value.microsecond,
            )
            # A time earlier than the start time may indicate a next-day end.
            if end < start:
                end += pd.Timedelta(days=1)
            return end

        if isinstance(value, (datetime, date, pd.Timestamp)):
            return pd.Timestamp(value)

        parsed = pd.to_datetime(value, errors="coerce", dayfirst=True)
        return pd.Timestamp(parsed) if not pd.isna(parsed) else pd.NaT

    numeric_ids = pd.to_numeric(raw["Attack #"], errors="coerce")
    numbered_mask = numeric_ids.notna() & (numeric_ids % 1 == 0)

    numbered = raw.loc[numbered_mask].copy()
    numbered["attack_id"] = numeric_ids.loc[numbered_mask].astype(int)

    start_times = numbered["Start Time"].map(parse_start)
    end_times = [
        parse_end(end, start)
        for end, start in zip(numbered["End Time"], start_times)
    ]

    attack_scenarios = pd.DataFrame({
        "attack_id": numbered["attack_id"].to_numpy(),
        "start_time": list(start_times),
        "end_time": end_times,
        "attack_points": numbered["Attack Point"].fillna("").astype(str).str.strip().to_numpy(),
        "start_state": numbered["Start State"].fillna("").astype(str).str.strip().to_numpy(),
        "attack": numbered["Attack"].fillna("").astype(str).str.strip().to_numpy(),
        "actual_change": numbered["Actual Change"].fillna("").astype(str).str.strip().to_numpy(),
        "description": numbered["Expected Impact or attacker intent"].fillna("").astype(str).str.strip().to_numpy(),
        "unexpected_outcome": numbered["Unexpected Outcome"].fillna("").astype(str).str.strip().to_numpy(),
    })

    # Keep non-numbered rows available for audit; do not interpret them as events.
    non_data_rows = raw.loc[~numbered_mask].copy()

    missing_start = int(attack_scenarios["start_time"].isna().sum())
    missing_end = int(attack_scenarios["end_time"].isna().sum())
    duplicate_ids = int(attack_scenarios["attack_id"].duplicated(keep=False).sum())
    invalid_intervals = int((
        attack_scenarios["start_time"].notna()
        & attack_scenarios["end_time"].notna()
        & (attack_scenarios["end_time"] < attack_scenarios["start_time"])
    ).sum())

    print(f"Total workbook rows: {len(raw)}")
    print(f"Numbered records: {len(attack_scenarios)}")
    print(f"Non-data/explanatory rows: {len(non_data_rows)}")
    print(f"Duplicate numbered IDs: {duplicate_ids}")
    print(f"Missing start times: {missing_start}")
    print(f"Missing end times: {missing_end}")
    print(f"End-before-start intervals: {invalid_intervals}")

    complete = attack_scenarios.loc[
        attack_scenarios["start_time"].notna()
        & attack_scenarios["end_time"].notna()
    ].sort_values("start_time").reset_index(drop=True)

    prior_max_end = complete["end_time"].cummax().shift(1)
    overlap_mask = (
        prior_max_end.notna()
        & (complete["start_time"] < prior_max_end)
    )
    print(f"Complete intervals overlapping a prior interval: {int(overlap_mask.sum())}")

    attack_scenarios.to_csv(
        NB09_OUT_DIR / "attack_scenarios_normalized.csv",
        index=False,
    )
    non_data_rows.to_csv(
        NB09_OUT_DIR / "attack_scenario_non_data_rows.csv",
        index=False,
    )

    HAVE_SCENARIOS = True
    print("Normalized scenario table and non-data rows saved.")


Total workbook rows: 49
Numbered records: 41
Non-data/explanatory rows: 8
Duplicate numbered IDs: 0
Missing start times: 0
Missing end times: 5
End-before-start intervals: 0
Complete intervals overlapping a prior interval: 0
Normalized scenario table and non-data rows saved.


In [9]:
# Purpose: Verify normalized scenario dates against the attack dataset.
# This cell checks alignment only; it does not assign scenario IDs.

attack_timestamps = pd.to_datetime(
    pd.Series(ATK_TIMESTAMPS),
    errors="coerce",
).dropna()

complete_scenarios = attack_scenarios.loc[
    attack_scenarios["start_time"].notna()
    & attack_scenarios["end_time"].notna()
].copy()

if attack_timestamps.empty:
    raise ValueError("No valid attack timestamps are available.")

if complete_scenarios.empty:
    raise ValueError("No complete scenario intervals are available.")

attack_start = attack_timestamps.min()
attack_end = attack_timestamps.max()

complete_scenarios["overlaps_attack_data"] = (
    (complete_scenarios["start_time"] <= attack_end)
    & (complete_scenarios["end_time"] >= attack_start)
)

print(f"Attack data range: {attack_start} -> {attack_end}")
print(f"Numbered scenarios: {len(attack_scenarios)}")
print(f"Complete scenarios: {len(complete_scenarios)}")
print(
    "Complete scenarios overlapping the attack data range: "
    f"{int(complete_scenarios['overlaps_attack_data'].sum())}"
)
print(
    "Complete scenarios outside the attack data range: "
    f"{int((~complete_scenarios['overlaps_attack_data']).sum())}"
)

display(
    complete_scenarios[
        [
            "attack_id",
            "start_time",
            "end_time",
            "attack_points",
            "overlaps_attack_data",
        ]
    ].sort_values("start_time")
)

Attack data range: 2015-12-28 10:29:43 -> 2016-01-02 13:41:11
Numbered scenarios: 41
Complete scenarios: 36
Complete scenarios overlapping the attack data range: 31
Complete scenarios outside the attack data range: 5


,attack_id,start_time,end_time,attack_points,overlaps_attack_data
36,37,2015-01-02 11:17:02,2015-01-02 11:24:50,"P-501, FIT-502",False
37,38,2015-01-02 11:31:38,2015-01-02 11:36:18,"AIT-402, AIT-502",False
38,39,2015-01-02 11:43:48,2015-01-02 11:50:28,"FIT-401, AIT-502",False
39,40,2015-01-02 11:51:42,2015-01-02 11:56:38,FIT-401,False
40,41,2015-01-02 13:13:02,2015-01-02 13:40:56,LIT-301,False
0,1,2015-12-28 10:29:14,2015-12-28 10:44:53,MV-101,True
1,2,2015-12-28 10:51:08,2015-12-28 10:58:30,P-102,True
2,3,2015-12-28 11:22:00,2015-12-28 11:28:22,LIT-101,True
3,4,2015-12-28 11:47:39,2015-12-28 11:54:08,MV-504,True
5,6,2015-12-28 12:00:55,2015-12-28 12:04:10,AIT-202,True


In [10]:
# Purpose: Audit the five out-of-range scenarios against the original workbook.

raw_scenarios = pd.read_excel(ATTACK_LIST_PATH, sheet_name="Sheet1")
raw_scenarios.columns = raw_scenarios.columns.astype(str).str.strip()

raw_scenarios["attack_id"] = pd.to_numeric(
    raw_scenarios["Attack #"], errors="coerce"
)

out_of_range_ids = [37, 38, 39, 40, 41]

raw_audit = raw_scenarios.loc[
    raw_scenarios["attack_id"].isin(out_of_range_ids),
    [
        "Attack #",
        "Start Time",
        "End Time",
        "Attack Point",
        "Attack",
    ],
].copy()

normalized_audit = attack_scenarios.loc[
    attack_scenarios["attack_id"].isin(out_of_range_ids),
    ["attack_id", "start_time", "end_time", "attack_points", "attack"],
].copy()

print("Original workbook values:")
display(raw_audit)

print("Normalized values:")
display(normalized_audit.sort_values("attack_id"))

Original workbook values:


,Attack #,Start Time,End Time,Attack Point,Attack
36,37,2015-01-02 11:17:02,11:24:50,"P-501, FIT-502",Close P-501; Set value of FIT-502 to 1.29 at ...
37,38,2015-01-02 11:31:38,11:36:18,"AIT-402, AIT-502",Set value of AIT402 as 260; Set value of AIT50...
38,39,2015-01-02 11:43:48,11:50:28,"FIT-401, AIT-502",Set value of FIT-401 as 0.5; Set value of AIT-...
39,40,2015-01-02 11:51:42,11:56:38,FIT-401,Set value of FIT-401 as 0
40,41,2015-01-02 13:13:02,13:40:56,LIT-301,decrease value by 0.5 mm per second


Normalized values:


,attack_id,start_time,end_time,attack_points,attack
36,37,2015-01-02 11:17:02,2015-01-02 11:24:50,"P-501, FIT-502",Close P-501; Set value of FIT-502 to 1.29 at ...
37,38,2015-01-02 11:31:38,2015-01-02 11:36:18,"AIT-402, AIT-502",Set value of AIT402 as 260; Set value of AIT50...
38,39,2015-01-02 11:43:48,2015-01-02 11:50:28,"FIT-401, AIT-502",Set value of FIT-401 as 0.5; Set value of AIT-...
39,40,2015-01-02 11:51:42,2015-01-02 11:56:38,FIT-401,Set value of FIT-401 as 0
40,41,2015-01-02 13:13:02,2015-01-02 13:40:56,LIT-301,decrease value by 0.5 mm per second


In [11]:

# Purpose: Inspect attack.csv during the five disputed scenario windows.
# Read in chunks to avoid loading the full CSV into memory.

from datetime import datetime
import pandas as pd

DISPUTED_IDS = [37, 38, 39, 40, 41]

disputed = attack_scenarios.loc[
    attack_scenarios["attack_id"].isin(DISPUTED_IDS),
    ["attack_id", "start_time", "end_time", "attack_points"],
].copy()

# Keep the original workbook dates unchanged.
disputed["comparison_start"] = disputed["start_time"].map(
    lambda ts: ts.replace(year=2016) if pd.notna(ts) else pd.NaT
)
disputed["comparison_end"] = disputed["end_time"].map(
    lambda ts: ts.replace(year=2016) if pd.notna(ts) else pd.NaT
)

if disputed[["comparison_start", "comparison_end"]].isna().any().any():
    raise ValueError("A disputed scenario has a missing start or end time.")

# Inspect the raw CSV schema before selecting columns.
header = pd.read_csv(ATTACK_CSV, nrows=0, skipinitialspace=True)
timestamp_column = next(
    (col for col in header.columns if str(col).strip().lower() == "timestamp"),
    None,
)
if timestamp_column is None:
    raise ValueError("Could not find the Timestamp column in attack.csv.")

matched_rows = []
rows_scanned = 0
invalid_timestamps = 0

for chunk in pd.read_csv(
    ATTACK_CSV,
    chunksize=100_000,
    skipinitialspace=True,
    low_memory=False,
):
    chunk.columns = chunk.columns.astype(str).str.strip()
    timestamps = pd.to_datetime(
        chunk[timestamp_column],
        errors="coerce",
        dayfirst=True,
    )

    rows_scanned += len(chunk)
    invalid_timestamps += int(timestamps.isna().sum())

    # Collect observations within any disputed scenario's proposed 2016 window.
    keep = pd.Series(False, index=chunk.index)
    for _, scenario in disputed.iterrows():
        keep |= (
            timestamps.ge(scenario["comparison_start"])
            & timestamps.le(scenario["comparison_end"])
        )

    if keep.any():
        selected = chunk.loc[keep].copy()
        selected["parsed_timestamp"] = timestamps.loc[keep].to_numpy()
        matched_rows.append(selected)

if matched_rows:
    disputed_observations = pd.concat(matched_rows, ignore_index=True)
else:
    disputed_observations = pd.DataFrame()

print(f"Attack CSV rows scanned: {rows_scanned:,}")
print(f"Invalid/unparseable timestamps: {invalid_timestamps:,}")
print(f"Rows within the five comparison windows: {len(disputed_observations):,}")

for _, scenario in disputed.iterrows():
    if disputed_observations.empty:
        window = disputed_observations
    else:
        window = disputed_observations.loc[
            disputed_observations["parsed_timestamp"].between(
                scenario["comparison_start"],
                scenario["comparison_end"],
            )
        ]

    print(
        f"Scenario {int(scenario['attack_id'])}: "
        f"{scenario['comparison_start']} -> "
        f"{scenario['comparison_end']} | "
        f"{len(window):,} rows"
    )

if not disputed_observations.empty:
    print("\nFirst and last observations in the inspected windows:")
    display(
        disputed_observations[
            ["parsed_timestamp"]
        ].agg(["min", "max"])
    )

    # Show a small sample of the observations and sensor values.
    display(disputed_observations.head(10))


Attack CSV rows scanned: 54,621
Invalid/unparseable timestamps: 0
Rows within the five comparison windows: 3,123
Scenario 37: 2016-01-02 11:17:02 -> 2016-01-02 11:24:50 | 469 rows
Scenario 38: 2016-01-02 11:31:38 -> 2016-01-02 11:36:18 | 281 rows
Scenario 39: 2016-01-02 11:43:48 -> 2016-01-02 11:50:28 | 401 rows
Scenario 40: 2016-01-02 11:51:42 -> 2016-01-02 11:56:38 | 297 rows
Scenario 41: 2016-01-02 13:13:02 -> 2016-01-02 13:40:56 | 1,675 rows

First and last observations in the inspected windows:


,parsed_timestamp
min,2016-01-02 11:17:02
max,2016-01-02 13:40:56


,Timestamp,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,...,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack,parsed_timestamp
0,2/1/2016 11:17:02 AM,2.611536,706.6282,2,1,1,169.8603,8.693155,291.1305,0.0,...,1,250.3365,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:02
1,2/1/2016 11:17:03 AM,2.649329,706.8636,2,1,1,169.8603,8.693155,291.1305,0.0,...,1,250.3365,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:03
2,2/1/2016 11:17:04 AM,2.668225,707.1384,2,1,1,169.8603,8.695719,291.1305,0.0,...,1,250.4326,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:04
3,2/1/2016 11:17:05 AM,2.676873,707.6095,2,1,1,169.8603,8.697641,291.1305,0.0,...,1,250.4326,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:05
4,2/1/2016 11:17:06 AM,2.674951,707.8842,2,1,1,169.8603,8.697641,291.1305,0.0,...,1,250.4326,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:06
5,2/1/2016 11:17:07 AM,2.643564,708.2768,2,1,1,169.8603,8.697641,291.1818,0.0,...,1,250.4326,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:07
6,2/1/2016 11:17:08 AM,2.634596,708.6693,2,1,1,169.8603,8.697641,291.1818,0.0,...,1,250.2563,0.961138,188.4453,0.0,1,1,1,Attack,2016-01-02 11:17:08
7,2/1/2016 11:17:09 AM,2.629152,708.9833,2,1,1,169.8603,8.702448,291.3868,0.0,...,1,250.2563,0.961138,188.3011,0.0,1,1,1,Attack,2016-01-02 11:17:09
8,2/1/2016 11:17:10 AM,2.618582,709.4543,2,1,1,169.8603,8.702448,291.3868,0.0,...,1,250.2563,0.961138,188.2850,0.0,1,1,1,Attack,2016-01-02 11:17:10
9,2/1/2016 11:17:11 AM,2.596804,709.8861,2,1,1,169.8603,8.702448,291.3868,0.0,...,1,250.2563,0.961138,188.2370,0.0,1,1,1,Attack,2016-01-02 11:17:11


In [13]:

# # Diagnose overlapping scenario windows without changing assignments.

# overlap_positions = np.flatnonzero(match_count > 1)

# overlap_details = []

# for position in overlap_positions:
#     endpoint_time = atk_seq_df.iloc[position]["timestamp"]
#     matching_scenarios = complete_scenarios.loc[
#         (complete_scenarios["start_time"] <= endpoint_time)
#         & (complete_scenarios["end_time"] >= endpoint_time)
#     ]

#     overlap_details.append({
#         "seq_index": int(atk_seq_df.iloc[position]["seq_index"]),
#         "end_index": int(atk_seq_df.iloc[position]["end_index"]),
#         "timestamp": endpoint_time,
#         "scenario_ids": matching_scenarios["attack_id"].tolist(),
#         "number_of_matching_scenarios": len(matching_scenarios),
#     })

# overlap_diagnostic = pd.DataFrame(overlap_details)

# display(overlap_diagnostic)

# if not overlap_diagnostic.empty:
#     overlapping_ids = sorted({
#         scenario_id
#         for ids in overlap_diagnostic["scenario_ids"]
#         for scenario_id in ids
#     })

#     display(
#         complete_scenarios.loc[
#             complete_scenarios["attack_id"].isin(overlapping_ids),
#             [
#                 "attack_id",
#                 "start_time",
#                 "end_time",
#                 "attack_points",
#                 "attack",
#                 "actual_change",
#             ],
#         ].sort_values(["start_time", "attack_id"])
#     )


In [14]:

# Purpose: Align attack sequence endpoints to documented scenario windows.
# Incomplete intervals are excluded; candidate date corrections are recorded
# separately and do not alter the original normalized scenario table.

atk_seq_df = pd.DataFrame({
    "seq_index": np.arange(len(attack_end_indices)),
    "end_index": attack_end_indices,
    "timestamp": pd.to_datetime(ATK_TIMESTAMPS, errors="raise"),
    "label": ATK_LABELS,
})

if not HAVE_SCENARIOS:
    raise RuntimeError(
        "Scenario data is unavailable. Resolve this before event matching."
    )

scen_for_matching = attack_scenarios.copy()

# Candidate matching assumption only: IDs 37–41 may belong to 2016-01-02.
candidate_ids = [37, 38, 39, 40, 41]
candidate_mask = scen_for_matching["attack_id"].isin(candidate_ids)

scen_for_matching.loc[candidate_mask, "start_time"] = (
    scen_for_matching.loc[candidate_mask, "start_time"]
    .map(lambda x: x.replace(year=2016) if pd.notna(x) else pd.NaT)
)
scen_for_matching.loc[candidate_mask, "end_time"] = (
    scen_for_matching.loc[candidate_mask, "end_time"]
    .map(lambda x: x.replace(year=2016) if pd.notna(x) else pd.NaT)
)

# Only complete, valid intervals can be used for timestamp matching.
complete_scenarios = scen_for_matching.loc[
    scen_for_matching["start_time"].notna()
    & scen_for_matching["end_time"].notna()
    & (
        scen_for_matching["end_time"]
        >= scen_for_matching["start_time"]
    )
].copy()

timestamps = atk_seq_df["timestamp"].to_numpy(dtype="datetime64[ns]")
match_count = np.zeros(len(atk_seq_df), dtype=np.int16)
scenario_ids = np.full(len(atk_seq_df), None, dtype=object)

for _, scenario in complete_scenarios.iterrows():
    in_window = (
        (timestamps >= np.datetime64(scenario["start_time"]))
        & (timestamps < np.datetime64(scenario["end_time"]))
    )
    match_count += in_window.astype(np.int16)
    scenario_ids[in_window] = scenario["attack_id"]

# Do not silently allow a sequence endpoint to belong to multiple scenarios.
if (match_count > 1).any():
    raise ValueError(
        f"{int((match_count > 1).sum())} endpoints match overlapping "
        "scenario windows. Resolve overlaps before event evaluation."
    )

atk_seq_df["scenario_id"] = scenario_ids
atk_seq_df["tcn_score"] = tcn_atk["attack_seq_scores"]
atk_seq_df["lstm_score"] = lstm_atk["attack_seq_scores"]
atk_seq_df["usad_score"] = usad_atk["attack_seq_scores"]

matched = int(atk_seq_df["scenario_id"].notna().sum())
matched_scenarios = int(
    atk_seq_df.loc[
        atk_seq_df["scenario_id"].notna(), "scenario_id"
    ].nunique()
)

print(f"Attack sequence endpoints: {len(atk_seq_df):,}")
print(f"Complete scenario intervals: {len(complete_scenarios)}")
print(f"Matched endpoints: {matched:,}")
print(f"Endpoint coverage: {matched / len(atk_seq_df) * 100:.2f}%")
print(f"Scenarios with matched endpoints: {matched_scenarios}")

# Save the matching assumptions and endpoint assignments for auditability.
scen_for_matching.to_csv(
    NB09_OUT_DIR / "attack_scenarios_matching_dates.csv",
    index=False,
)
atk_seq_df.to_parquet(
    NB09_OUT_DIR / "attack_sequences_with_scenarios.parquet",
    index=False,
)

assert len(atk_seq_df) == 54592
assert "scenario_id" in atk_seq_df.columns
assert len(atk_seq_df["tcn_score"]) == len(atk_seq_df)
assert len(atk_seq_df["lstm_score"]) == len(atk_seq_df)
assert len(atk_seq_df["usad_score"]) == len(atk_seq_df)

print("Scenario endpoint assignment completed.")
print(
    "Caution: date corrections for IDs 37–41 are candidate assumptions "
    "for matching, not verified corrections to the source workbook."
)


Attack sequence endpoints: 54,592
Complete scenario intervals: 36
Matched endpoints: 53,821
Endpoint coverage: 98.59%
Scenarios with matched endpoints: 36
Scenario endpoint assignment completed.
Caution: date corrections for IDs 37–41 are candidate assumptions for matching, not verified corrections to the source workbook.


### Scenario Alignment — Findings and Decisions

- **Scenario source:** The attack scenario workbook contains 41 numbered records. Five records have incomplete time intervals and are excluded from timestamp matching, leaving 36 complete intervals.
- **Timestamp discrepancy:** Scenarios 37–41 are dated 2 January 2015 in the source workbook, outside the attack dataset's date range. Testing their month, day and times against 2 January 2016 found attack-labelled observations within all five candidate windows.
- **Matching decision:** The five dates are adjusted to 2016 in a derived matching table only. This is a documented working assumption, not a verified correction to the original workbook. The source workbook and normalized scenario table remain unchanged.
- **Boundary handling:** Scenario intervals use start-inclusive, end-exclusive matching. This prevents an endpoint at a shared boundary from being assigned to two consecutive scenarios.
- **Assignment results:** Of 54,592 attack sequence endpoints, 53,821 (98.59%) matched a documented scenario window. All 36 included scenarios matched at least one endpoint, and no overlapping assignments remained.
- **Interpretation:** The 98.59% figure measures endpoint coverage by scenario windows, not anomaly-detection performance. The 771 unmatched endpoints remain outside the assigned scenario windows and should not automatically be treated as errors.
- **Limitation:** The intended dates for scenarios 37–41 remain unverified and must be reported as an assumption in subsequent event analysis.

In [15]:
# Purpose: Compute predictions at the frozen thresholds for all 9 methods × 3 partitions.
#          LINCHPIN ASSERT: TP/FP/TN/FN must match evaluation_summary.csv exactly for every method.
eval_summary = pd.read_csv(NB08_DIR / "evaluation_summary.csv").set_index("Method")
assert set(eval_summary.index) == set(METHODS), f"evaluation_summary methods mismatch: {set(eval_summary.index)}"

PRED = {}
summary_rows = []
mismatches   = []
for m in METHODS:
    thr   = float(THRESHOLDS[m])
    s_thr = SEQ_SCORES[f"{m}_threshold_seq_scores"]
    s_val = SEQ_SCORES[f"{m}_validation_seq_scores"]
    s_atk = SEQ_SCORES[f"{m}_attack_seq_scores"]

    PRED[(m, "threshold")]  = (s_thr > thr).astype(np.int8)
    PRED[(m, "validation")] = (s_val > thr).astype(np.int8)
    PRED[(m, "attack")]     = (s_atk > thr).astype(np.int8)

    TP = int(PRED[(m, "attack")].sum());     FN = int(len(s_atk) - TP)
    FP = int(PRED[(m, "validation")].sum()); TN = int(len(s_val) - FP)

    exp_TP = int(eval_summary.loc[m, "TP"]); exp_FP = int(eval_summary.loc[m, "FP"])
    exp_TN = int(eval_summary.loc[m, "TN"]); exp_FN = int(eval_summary.loc[m, "FN"])

    ok = (TP == exp_TP) and (FP == exp_FP) and (TN == exp_TN) and (FN == exp_FN)
    if not ok:
        mismatches.append(
            f"{m}: got TP={TP} FP={FP} TN={TN} FN={FN} | expected TP={exp_TP} FP={exp_FP} TN={exp_TN} FN={exp_FN}")

    summary_rows.append({"Method": m, "TP": TP, "FP": FP, "TN": TN, "FN": FN,
                         "Threshold": thr, "Match_NB08": ok})

if mismatches:
    raise AssertionError("LINCHPIN MISMATCH vs evaluation_summary.csv:\n" + "\n".join(mismatches))

PRED_VERIFY = pd.DataFrame(summary_rows).set_index("Method")
assert PRED_VERIFY["Match_NB08"].all()
print("LINCHPIN PASS: all 9 methods match evaluation_summary TP/FP/TN/FN exactly")
print(PRED_VERIFY[["TP","FP","TN","FN"]].to_string())

LINCHPIN PASS: all 9 methods match evaluation_summary TP/FP/TN/FN exactly
            TP     FP     TN     FN
Method                             
rule     47509  73295   5736   7083
rolling  19987  32669  46362  34605
simple   44933  10823  68208   9659
if       33236   1155  77876  21356
ocsvm    34529    650  78381  20063
lof       3355   2147  76884  51237
lstm     41046   2259  76772  13546
tcn      43345   2528  76503  11247
usad     35135   1261  77770  19457


In [16]:
# Purpose: Build point-level error_inventory table with Precision/Recall/F1/FPR.

error_inventory = PRED_VERIFY.copy()

precision_den = error_inventory["TP"] + error_inventory["FP"]
recall_den = error_inventory["TP"] + error_inventory["FN"]
fpr_den = error_inventory["FP"] + error_inventory["TN"]

error_inventory["Precision"] = (
    error_inventory["TP"] / precision_den.replace(0, np.nan)
)

error_inventory["Recall"] = (
    error_inventory["TP"] / recall_den.replace(0, np.nan)
)

f1_den = error_inventory["Precision"] + error_inventory["Recall"]

error_inventory["F1"] = (
    2
    * error_inventory["Precision"]
    * error_inventory["Recall"]
    / f1_den.replace(0, np.nan)
)

error_inventory["FPR"] = (
    error_inventory["FP"] / fpr_den.replace(0, np.nan)
)

error_inventory.to_csv(
    NB09_OUT_DIR / "error_inventory.csv"
)

print(
    error_inventory.sort_values("F1", ascending=False)[
        [
            "TP",
            "FP",
            "TN",
            "FN",
            "Precision",
            "Recall",
            "F1",
            "FPR",
        ]
    ].round(4).to_string()
)

# Structural checks.
assert error_inventory.shape[0] == 9
assert (
    error_inventory["TP"] + error_inventory["FN"] == 54592
).all()
assert (
    error_inventory["FP"] + error_inventory["TN"] == 79031
).all()

# Metric validity checks.
assert error_inventory[
    ["Precision", "Recall", "F1", "FPR"]
].apply(lambda col: col.between(0, 1).all()).all()

print("\nPoint-level error inventory created and validated.")

            TP     FP     TN     FN  Precision  Recall      F1     FPR
Method                                                                
tcn      43345   2528  76503  11247     0.9449  0.7940  0.8629  0.0320
lstm     41046   2259  76772  13546     0.9478  0.7519  0.8386  0.0286
simple   44933  10823  68208   9659     0.8059  0.8231  0.8144  0.1369
usad     35135   1261  77770  19457     0.9654  0.6436  0.7723  0.0160
ocsvm    34529    650  78381  20063     0.9815  0.6325  0.7693  0.0082
if       33236   1155  77876  21356     0.9664  0.6088  0.7470  0.0146
rule     47509  73295   5736   7083     0.3933  0.8703  0.5417  0.9274
rolling  19987  32669  46362  34605     0.3796  0.3661  0.3727  0.4134
lof       3355   2147  76884  51237     0.6098  0.0615  0.1117  0.0272

Point-level error inventory created and validated.


### Point-Level Error Analysis — Findings

- TCN achieved the highest point-level F1 score among the nine methods (F1 = 0.8629), with precision of 0.9449, recall of 0.7940, and FPR of 0.0320.

- LSTM ranked second by F1 (0.8386), with precision of 0.9478, recall of 0.7519, and FPR of 0.0286.

- The simple statistical threshold achieved F1 = 0.8144, with a higher recall (0.8231) but substantially higher FPR (0.1369) than TCN and LSTM.

- USAD, One-Class SVM, and Isolation Forest produced relatively high precision and low FPR but lower recall than TCN, indicating that they missed more attack endpoints at their frozen thresholds.

- The rule-based method produced high recall (0.8703) but an extremely high FPR (0.9274), while the rolling method showed weak precision and recall.

- LOF had the lowest point-level F1 (0.1117), primarily due to very low recall (0.0615).

- These are point-level results only. They do not establish which method detects complete attack events most effectively. Event coverage, detection delay, and false-alarm behavior will be examined separately.

#### The simple statistical threshold showed a different operating point from the learned models, with higher recall but substantially higher false-positive rate. The reason for this behavior is not inferred from point-level metrics alone and will be examined through event-level analysis.

In [17]:
# Purpose: Build the long-form all-predictions table
#          (9 methods × 3 partitions) with sequence-endpoint timestamps.

timestamps_path = NB09_OUT_DIR / "derived_timestamps.npz"

with np.load(timestamps_path) as ts_data:
    thr_ts_arr = pd.to_datetime(ts_data["threshold_ts"])
    val_ts_arr = pd.to_datetime(ts_data["validation_ts"])

atk_ts_arr = pd.to_datetime(ATK_TIMESTAMPS)
atk_scenario_ids = atk_seq_df["scenario_id"].to_numpy()

# Verify timestamp and scenario arrays match the attack sequence partition.
assert len(thr_ts_arr) == len(SEQ_SCORES["rule_threshold_seq_scores"]), (
    f"Threshold timestamp count mismatch: "
    f"{len(thr_ts_arr)} timestamps vs "
    f"{len(SEQ_SCORES['rule_threshold_seq_scores'])} threshold sequences."
)

assert len(val_ts_arr) == len(SEQ_SCORES["rule_validation_seq_scores"]), (
    f"Validation timestamp count mismatch: "
    f"{len(val_ts_arr)} timestamps vs "
    f"{len(SEQ_SCORES['rule_validation_seq_scores'])} validation sequences."
)

assert len(atk_ts_arr) == len(SEQ_SCORES["rule_attack_seq_scores"]), (
    f"Attack timestamp count mismatch: "
    f"{len(atk_ts_arr)} timestamps vs "
    f"{len(SEQ_SCORES['rule_attack_seq_scores'])} attack sequences."
)

assert len(atk_scenario_ids) == len(atk_ts_arr), (
    f"Attack scenario_id count mismatch: "
    f"{len(atk_scenario_ids)} scenario IDs vs "
    f"{len(atk_ts_arr)} attack timestamps."
)

frames = []

for m in METHODS:
    threshold_scores = SEQ_SCORES[f"{m}_threshold_seq_scores"]
    validation_scores = SEQ_SCORES[f"{m}_validation_seq_scores"]
    attack_scores = SEQ_SCORES[f"{m}_attack_seq_scores"]

    # Verify each method has the expected sequence counts.
    assert len(threshold_scores) == len(thr_ts_arr)
    assert len(validation_scores) == len(val_ts_arr)
    assert len(attack_scores) == len(atk_ts_arr)

    frames.append(
        pd.DataFrame({
            "method": m,
            "partition": "threshold",
            "seq_index": np.arange(len(threshold_scores)),
            "timestamp": thr_ts_arr,
            "score": threshold_scores,
            "prediction": PRED[(m, "threshold")],
            "label": 0,
            "scenario_id": None,
        })
    )

    frames.append(
        pd.DataFrame({
            "method": m,
            "partition": "validation",
            "seq_index": np.arange(len(validation_scores)),
            "timestamp": val_ts_arr,
            "score": validation_scores,
            "prediction": PRED[(m, "validation")],
            "label": 0,
            "scenario_id": None,
        })
    )

    frames.append(
        pd.DataFrame({
            "method": m,
            "partition": "attack",
            "seq_index": np.arange(len(attack_scores)),
            "timestamp": atk_ts_arr,
            "score": attack_scores,
            "prediction": PRED[(m, "attack")],
            "label": 1,
            "scenario_id": atk_scenario_ids,
        })
    )

PRED_DF = pd.concat(frames, ignore_index=True)

PRED_DF["prediction"] = PRED_DF["prediction"].astype(np.int8)

expected_rows = 9 * (
    len(thr_ts_arr) +
    len(val_ts_arr) +
    len(atk_ts_arr)
)

assert len(PRED_DF) == expected_rows
assert len(PRED_DF) == 1_629_279, (
    f"Unexpected total row count: {len(PRED_DF)}"
)

assert set(PRED_DF["method"].unique()) == set(METHODS)
assert set(PRED_DF["partition"].unique()) == {
    "threshold", "validation", "attack"
}
assert set(PRED_DF["prediction"].unique()).issubset({0, 1})
assert PRED_DF["score"].notna().all()
assert PRED_DF["timestamp"].notna().all()

# Verify partition-level row counts.
for partition, expected_count in {
    "threshold": 9 * len(thr_ts_arr),
    "validation": 9 * len(val_ts_arr),
    "attack": 9 * len(atk_ts_arr),
}.items():
    actual_count = int((PRED_DF["partition"] == partition).sum())
    assert actual_count == expected_count, (
        f"{partition} row count mismatch: "
        f"{actual_count} vs {expected_count}"
    )

PRED_DF.to_parquet(
    NB09_OUT_DIR / "all_predictions.parquet",
    index=False,
)

print(f"PRED_DF rows = {len(PRED_DF):,}")
print(f"Expected rows = {expected_rows:,}")
print("Methods:", ", ".join(METHODS))
print("Partitions:", ", ".join(sorted(PRED_DF["partition"].unique())))
print("All-predictions table created and validated.")

PRED_DF rows = 1,629,279
Expected rows = 1,629,279
Methods: rule, rolling, simple, if, ocsvm, lof, lstm, tcn, usad
Partitions: attack, threshold, validation
All-predictions table created and validated.


In [19]:
# Purpose: Audit scenario records before event-level analysis.

print("Scenario table shape:", scen_for_matching.shape)
print("\nColumns:")
print(scen_for_matching.columns.tolist())

print("\nScenario records:")
display(scen_for_matching.to_string(index=False))

print("\nMissing values by column:")
print(scen_for_matching.isna().sum().to_string())

Scenario table shape: (41, 9)

Columns:
['attack_id', 'start_time', 'end_time', 'attack_points', 'start_state', 'attack', 'actual_change', 'description', 'unexpected_outcome']

Scenario records:


' attack_id          start_time            end_time             attack_points                                                                                         start_state                                                                                                                                        attack actual_change                                                                                                                                   description                                                                                               unexpected_outcome\n         1 2015-12-28 10:29:14 2015-12-28 10:44:53                    MV-101                                                                                    MV-101 is closed                                                                                                                                   Open MV-101           Yes                                                                              


Missing values by column:
attack_id             0
start_time            0
end_time              5
attack_points         0
start_state           0
attack                0
actual_change         0
description           0
unexpected_outcome    0


In [20]:

# Purpose: Evaluate scenario-level detection coverage and first-detection
#          delay for all 9 methods across the 36 complete attack scenarios.

# 1. Prepare the scenario table and exclude incomplete intervals.
scenario_table = scen_for_matching[
    ["attack_id", "start_time", "end_time"]
].copy()

scenario_table = scenario_table.rename(
    columns={
        "attack_id": "scenario_id",
        "start_time": "scenario_start",
        "end_time": "scenario_end",
    }
)

scenario_table["scenario_start"] = pd.to_datetime(
    scenario_table["scenario_start"], errors="coerce"
)
scenario_table["scenario_end"] = pd.to_datetime(
    scenario_table["scenario_end"], errors="coerce"
)

incomplete_scenarios = scenario_table.loc[
    scenario_table["scenario_start"].isna()
    | scenario_table["scenario_end"].isna(),
    "scenario_id",
].tolist()

scenario_table = scenario_table.loc[
    scenario_table["scenario_start"].notna()
    & scenario_table["scenario_end"].notna()
].copy()

scenario_table["scenario_id"] = pd.to_numeric(
    scenario_table["scenario_id"], errors="raise"
).astype(int)

assert len(incomplete_scenarios) == 5, (
    f"Expected 5 incomplete records; found {len(incomplete_scenarios)}."
)
assert set(incomplete_scenarios) == {5, 9, 12, 15, 18}
assert len(scenario_table) == 36, (
    f"Expected 36 complete scenarios; found {len(scenario_table)}."
)
assert scenario_table["scenario_id"].is_unique
assert (
    scenario_table["scenario_start"] < scenario_table["scenario_end"]
).all()

# 2. Select attack-partition predictions with existing scenario assignments.
attack_predictions = PRED_DF.loc[
    PRED_DF["partition"] == "attack"
].copy()

attack_predictions["scenario_id"] = pd.to_numeric(
    attack_predictions["scenario_id"], errors="coerce"
).astype("Int64")

matched_predictions = attack_predictions.loc[
    attack_predictions["scenario_id"].notna()
].copy()

valid_scenario_ids = set(scenario_table["scenario_id"])
assigned_scenario_ids = set(
    matched_predictions["scenario_id"].astype(int).unique()
)

assert assigned_scenario_ids.issubset(valid_scenario_ids), (
    "Predictions contain scenario IDs absent from the complete scenario table."
)

# 3. Confirm scenario assignments are identical across methods.
reference_assignments = (
    matched_predictions.loc[
        matched_predictions["method"] == METHODS[0]
    ]
    .sort_values("seq_index")["scenario_id"]
    .reset_index(drop=True)
)

for method in METHODS:
    method_assignments = (
        matched_predictions.loc[
            matched_predictions["method"] == method
        ]
        .sort_values("seq_index")["scenario_id"]
        .reset_index(drop=True)
    )

    assert method_assignments.equals(reference_assignments), (
        f"Scenario assignments differ for method: {method}"
    )

# 4. Calculate detection results for every method and complete scenario.
scenario_rows = []

for _, scenario in scenario_table.iterrows():
    scenario_id = int(scenario["scenario_id"])
    scenario_start = scenario["scenario_start"]
    scenario_end = scenario["scenario_end"]

    scenario_predictions = matched_predictions.loc[
        matched_predictions["scenario_id"] == scenario_id
    ]

    for method in METHODS:
        method_predictions = (
            scenario_predictions.loc[
                scenario_predictions["method"] == method
            ]
            .sort_values("timestamp")
        )

        total_endpoints = len(method_predictions)

        detected = method_predictions.loc[
            method_predictions["prediction"] == 1
        ]

        detected_endpoints = len(detected)
        any_detection = detected_endpoints > 0

        if any_detection:
            first_detection = detected["timestamp"].iloc[0]
            delay_seconds = (
                first_detection - scenario_start
            ).total_seconds()
        else:
            first_detection = pd.NaT
            delay_seconds = np.nan

        scenario_rows.append({
            "Method": method,
            "scenario_id": scenario_id,
            "scenario_start": scenario_start,
            "scenario_end": scenario_end,
            "scenario_endpoints": total_endpoints,
            "detected_endpoints": detected_endpoints,
            "detection_coverage": (
                detected_endpoints / total_endpoints
                if total_endpoints > 0 else np.nan
            ),
            "any_detection": any_detection,
            "first_detection_timestamp": first_detection,
            "detection_delay_seconds": delay_seconds,
            "detection_delay_minutes": (
                delay_seconds / 60
                if pd.notna(delay_seconds) else np.nan
            ),
            "missed_scenario": not any_detection,
            "date_adjustment_assumption": (
                scenario_id in {37, 38, 39, 40, 41}
            ),
        })

scenario_results = pd.DataFrame(scenario_rows)

# 5. Validate and save the detailed results.
assert len(scenario_results) == 9 * 36 == 324
assert scenario_results.groupby("Method")["scenario_id"].nunique().eq(36).all()
assert scenario_results["scenario_endpoints"].gt(0).all()
assert scenario_results["detection_coverage"].between(0, 1).all()
assert (
    scenario_results["detected_endpoints"]
    <= scenario_results["scenario_endpoints"]
).all()
assert (
    scenario_results["any_detection"]
    == ~scenario_results["missed_scenario"]
).all()

scenario_results.to_csv(
    NB09_OUT_DIR / "scenario_detection_results.csv",
    index=False,
)

# 6. Summarize results by method.
scenario_summary = (
    scenario_results.groupby("Method")
    .agg(
        scenarios=("scenario_id", "nunique"),
        scenarios_detected=("any_detection", "sum"),
        scenarios_missed=("missed_scenario", "sum"),
        mean_scenario_coverage=("detection_coverage", "mean"),
        median_detection_delay_minutes=("detection_delay_minutes", "median"),
    )
)

scenario_summary["scenario_detection_rate"] = (
    scenario_summary["scenarios_detected"]
    / scenario_summary["scenarios"]
)

scenario_summary.to_csv(
    NB09_OUT_DIR / "scenario_detection_summary.csv"
)

print("Incomplete scenario IDs excluded:", incomplete_scenarios)
print("Complete scenarios:", len(scenario_table))
print("Detailed method-scenario rows:", len(scenario_results))
print("\nScenario detection summary:")
print(scenario_summary.round(4).to_string())


Incomplete scenario IDs excluded: [5, 9, 12, 15, 18]
Complete scenarios: 36
Detailed method-scenario rows: 324

Scenario detection summary:
         scenarios  scenarios_detected  scenarios_missed  mean_scenario_coverage  median_detection_delay_minutes  scenario_detection_rate
Method                                                                                                                                   
if              36                  13                23                  0.1074                          1.8500                   0.3611
lof             36                  15                21                  0.1496                          1.3667                   0.4167
lstm            36                  30                 6                  0.4292                          0.0250                   0.8333
ocsvm           36                   8                28                  0.1510                          0.3417                   0.2222
rolling         36              

### Scenario-Level Detection — Findings

The scenario-level evaluation covered 36 complete attack intervals across nine detection methods. Five records with incomplete intervals (scenarios 5, 9, 12, 15, and 18) were excluded.

Simple Statistical Threshold detected at least one anomalous sequence endpoint in 33 of 36 scenarios (91.67%), with a mean scenario coverage of 56.97%. TCN detected 28 scenarios (77.78%) and achieved 53.88% mean coverage, while LSTM detected 30 scenarios (83.33%) with 42.92% mean coverage.

The results show that Simple Statistical Threshold provides broader scenario detection than either deep-learning model under the current evaluation setup. However, its point-level false-positive rate of 13.69% is substantially higher than TCN's 3.20% and LSTM's 2.86%. Its higher scenario coverage therefore comes with a greater false-alarm burden.

Rule-Based Detection and Rolling Statistical Detection registered at least one detection in all 36 scenarios. This result alone does not establish superior performance: their mean scenario coverage was 74.63% and 40.99%, respectively, and their point-level false-positive rates were 92.74% and 41.34%.

These findings demonstrate that scenario detection rate, scenario coverage, and point-level false-positive rate measure different aspects of performance. Detection delay is measured from the documented scenario start to the first predicted anomalous endpoint and should not be interpreted as the time to confirmed physical process impact.

The results do not, by themselves, establish why Simple Statistical Threshold performs as it does. Further analysis of scenario-specific detections and the temporal concentration of validation false positives is required before drawing that conclusion. The date adjustment for scenarios 37–41 remains an explicit matching assumption.

In [21]:
# Purpose: Compare scenario-level detections across methods and
#          identify which scenarios distinguish Simple, TCN, and LSTM.

# 1. Create a per-scenario detection matrix for all methods.
detection_matrix = scenario_results.pivot(
    index="scenario_id",
    columns="Method",
    values="any_detection",
).reindex(columns=METHODS)

coverage_matrix = scenario_results.pivot(
    index="scenario_id",
    columns="Method",
    values="detection_coverage",
).reindex(columns=METHODS)

assert detection_matrix.shape == (36, 9)
assert detection_matrix.notna().all().all()
assert coverage_matrix.shape == (36, 9)

detection_matrix = detection_matrix.astype(bool)

detection_matrix.to_csv(
    NB09_OUT_DIR / "scenario_detection_matrix.csv"
)
coverage_matrix.to_csv(
    NB09_OUT_DIR / "scenario_coverage_matrix.csv"
)

# 2. Compare Simple, TCN, and LSTM scenario detection directly.
comparison = pd.DataFrame(index=detection_matrix.index)

for method in ["simple", "tcn", "lstm"]:
    comparison[f"{method}_detected"] = detection_matrix[method]
    comparison[f"{method}_coverage"] = coverage_matrix[method]

comparison["simple_only_vs_tcn_lstm"] = (
    detection_matrix["simple"]
    & ~detection_matrix["tcn"]
    & ~detection_matrix["lstm"]
)

comparison["tcn_or_lstm_missed_by_simple"] = (
    ~detection_matrix["simple"]
    & (detection_matrix["tcn"] | detection_matrix["lstm"])
)

comparison["all_three_detected"] = (
    detection_matrix["simple"]
    & detection_matrix["tcn"]
    & detection_matrix["lstm"]
)

comparison.to_csv(
    NB09_OUT_DIR / "simple_tcn_lstm_scenario_comparison.csv"
)

print("SCENARIO DETECTION COMPARISON")
print("-----------------------------")
print("Simple detects, TCN misses:",
      int((detection_matrix["simple"] & ~detection_matrix["tcn"]).sum()))
print("Simple detects, LSTM misses:",
      int((detection_matrix["simple"] & ~detection_matrix["lstm"]).sum()))
print("TCN detects, Simple misses:",
      int((detection_matrix["tcn"] & ~detection_matrix["simple"]).sum()))
print("LSTM detects, Simple misses:",
      int((detection_matrix["lstm"] & ~detection_matrix["simple"]).sum()))
print("All three detect:",
      int(comparison["all_three_detected"].sum()))
print("Detected only by Simple among these three:",
      int(comparison["simple_only_vs_tcn_lstm"].sum()))
print("Detected by TCN or LSTM but missed by Simple:",
      int(comparison["tcn_or_lstm_missed_by_simple"].sum()))

print("\nSCENARIOS DETECTED BY SIMPLE BUT MISSED BY BOTH TCN AND LSTM")
print("------------------------------------------------------------")
display(
    comparison.loc[
        comparison["simple_only_vs_tcn_lstm"],
        [
            "simple_detected", "simple_coverage",
            "tcn_detected", "tcn_coverage",
            "lstm_detected", "lstm_coverage",
        ],
    ].round(4)
)

print("\nSCENARIOS DETECTED BY TCN OR LSTM BUT MISSED BY SIMPLE")
print("------------------------------------------------------")
display(
    comparison.loc[
        comparison["tcn_or_lstm_missed_by_simple"],
        [
            "simple_detected", "simple_coverage",
            "tcn_detected", "tcn_coverage",
            "lstm_detected", "lstm_coverage",
        ],
    ].round(4)
)

print("\nAll-method scenario detection matrix (1 = at least one detection)")
display(detection_matrix.astype(int))


SCENARIO DETECTION COMPARISON
-----------------------------
Simple detects, TCN misses: 7
Simple detects, LSTM misses: 5
TCN detects, Simple misses: 2
LSTM detects, Simple misses: 2
All three detect: 24
Detected only by Simple among these three: 3
Detected by TCN or LSTM but missed by Simple: 2

SCENARIOS DETECTED BY SIMPLE BUT MISSED BY BOTH TCN AND LSTM
------------------------------------------------------------


,simple_detected,simple_coverage,tcn_detected,tcn_coverage,lstm_detected,lstm_coverage
scenario_id,,,,,,
14,True,0.0349,False,0.0,False,0.0
21,True,0.0333,False,0.0,False,0.0
25,True,0.0573,False,0.0,False,0.0



SCENARIOS DETECTED BY TCN OR LSTM BUT MISSED BY SIMPLE
------------------------------------------------------


,simple_detected,simple_coverage,tcn_detected,tcn_coverage,lstm_detected,lstm_coverage
scenario_id,,,,,,
27,False,0.0,True,0.0018,True,0.0124
29,False,0.0,True,0.2250,True,0.2333



All-method scenario detection matrix (1 = at least one detection)


Method,rule,rolling,simple,if,ocsvm,lof,lstm,tcn,usad
scenario_id,,,,,,,,,
1,1,1,1,0,0,1,1,0,0
2,1,1,1,1,0,1,1,1,1
3,1,1,1,0,0,0,0,1,0
4,1,1,1,0,0,0,1,0,1
6,1,1,1,0,0,0,1,1,0
7,1,1,1,0,0,0,1,1,0
8,1,1,1,1,0,0,1,1,1
10,1,1,1,0,1,1,1,1,1
11,1,1,1,1,1,1,1,1,1


In [22]:

# Purpose: Reconcile scenario-level summary counts with the detection matrix.

matrix_counts = detection_matrix.sum().rename("matrix_scenarios_detected")
summary_counts = scenario_summary["scenarios_detected"].rename(
    "summary_scenarios_detected"
)

scenario_count_audit = pd.concat(
    [matrix_counts, summary_counts], axis=1
)

scenario_count_audit["difference"] = (
    scenario_count_audit["matrix_scenarios_detected"]
    - scenario_count_audit["summary_scenarios_detected"]
)

print(scenario_count_audit.to_string())

assert (scenario_count_audit["difference"] == 0).all(), (
    "Scenario detection counts disagree. Rebuild the summary from "
    "the current scenario_results before proceeding."
)

print("\nPASS: scenario detection counts agree across both outputs.")


         matrix_scenarios_detected  summary_scenarios_detected  difference
Method                                                                    
rule                            36                          36           0
rolling                         36                          36           0
simple                          33                          33           0
if                              13                          13           0
ocsvm                            8                           8           0
lof                             15                          15           0
lstm                            30                          30           0
tcn                             28                          28           0
usad                            24                          24           0

PASS: scenario detection counts agree across both outputs.


In [24]:
# Purpose:
# Audit the validation timestamp cadence and quantify false-positive
# observations and consecutive false-alarm periods for all nine methods.
#
# This cell is diagnostic only. It does not change thresholds or predictions.

from pathlib import Path
import numpy as np
import pandas as pd

# 1. Validate the required data and output directory

required_columns = {
    "method",
    "partition",
    "timestamp",
    "label",
    "prediction",
}

missing_columns = required_columns.difference(PRED_DF.columns)

if missing_columns:
    raise KeyError(
        f"PRED_DF is missing required columns: {sorted(missing_columns)}"
    )

output_dir = Path(PROJECT_ROOT) / "nb09_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

validation = PRED_DF.loc[
    PRED_DF["partition"].eq("validation")
].copy()

if validation.empty:
    raise ValueError("No validation-partition predictions were found.")

validation["timestamp"] = pd.to_datetime(
    validation["timestamp"], errors="coerce"
)

if validation["timestamp"].isna().any():
    raise ValueError(
        "Validation contains missing or invalid timestamps. "
        "Resolve this before calculating false-alarm periods."
    )

if validation["label"].isna().any():
    raise ValueError("Validation contains missing labels.")

if validation["prediction"].isna().any():
    raise ValueError("Validation contains missing predictions.")

# Confirm that every method has one row per validation timestamp.
duplicate_rows = validation.duplicated(
    subset=["method", "timestamp"], keep=False
)

if duplicate_rows.any():
    raise ValueError(
        "Duplicate method/timestamp rows found in validation. "
        "Inspect these rows before proceeding."
    )

observed_methods = set(validation["method"].unique())
expected_methods = set(METHODS)

if observed_methods != expected_methods:
    raise ValueError(
        f"Unexpected method set. Expected {sorted(expected_methods)}, "
        f"found {sorted(observed_methods)}."
    )

# 2. Infer the observed validation timestamp cadence

unique_timestamps = pd.DatetimeIndex(
    validation["timestamp"].drop_duplicates().sort_values()
)

if len(unique_timestamps) < 2:
    raise ValueError("Insufficient unique timestamps to infer cadence.")

timestamp_gaps = (
    unique_timestamps.to_series()
    .diff()
    .dt.total_seconds()
    .dropna()
)

positive_gaps = timestamp_gaps[timestamp_gaps > 0]

if positive_gaps.empty:
    raise ValueError("Could not identify positive timestamp intervals.")

gap_counts = positive_gaps.round(6).value_counts()
expected_cadence_seconds = float(gap_counts.index[0])

cadence_frequency = int(gap_counts.iloc[0])
cadence_proportion = cadence_frequency / len(positive_gaps)

print("VALIDATION TIMESTAMP AUDIT")
print("-" * 60)
print(f"Unique timestamps: {len(unique_timestamps):,}")
print(f"First timestamp:   {unique_timestamps.min()}")
print(f"Last timestamp:    {unique_timestamps.max()}")
print(
    "Most frequent positive interval: "
    f"{expected_cadence_seconds:g} seconds"
)
print(
    "Share of intervals matching the most frequent interval: "
    f"{cadence_proportion:.2%}"
)
print("\nMost frequent observed intervals (seconds):")
print(gap_counts.head(10).to_string())

if cadence_proportion < 0.50:
    print(
        "\nWARNING: The most frequent interval accounts for less than "
        "half of all intervals. Treat the inferred cadence cautiously "
        "and inspect the timestamp gaps before interpreting periods."
    )

# A gap greater than 1.5 times the inferred cadence breaks a period.
# A non-false-positive row also breaks a period.
period_gap_limit_seconds = expected_cadence_seconds * 1.5

# 3. Calculate false-positive observations and periods

summary_rows = []
period_rows = []

for method in METHODS:
    method_df = (
        validation.loc[validation["method"].eq(method)]
        .sort_values("timestamp")
        .copy()
        .reset_index(drop=True)
    )

    # A false positive is a predicted anomaly on a normal-labelled row.
    is_false_positive = (
        method_df["label"].eq(0)
        & method_df["prediction"].eq(1)
    )

    normal_count = int(method_df["label"].eq(0).sum())
    false_positive_count = int(is_false_positive.sum())

    false_positive_rate = (
        false_positive_count / normal_count
        if normal_count > 0
        else np.nan
    )

    method_df["is_false_positive"] = is_false_positive
    method_df["gap_from_previous_seconds"] = (
        method_df["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    previous_is_false_positive = (
        method_df["is_false_positive"]
        .shift(1, fill_value=False)
    )

    starts_new_period = (
        method_df["is_false_positive"]
        & (
            ~previous_is_false_positive
            | method_df["gap_from_previous_seconds"].gt(
                period_gap_limit_seconds
            )
        )
    )

    method_df["period_id"] = starts_new_period.cumsum()

    method_periods = method_df.loc[
        method_df["is_false_positive"]
    ].groupby("period_id", sort=True)

    method_period_count = 0
    method_period_durations = []

    for period_id, period in method_periods:
        start_time = period["timestamp"].iloc[0]
        end_time = period["timestamp"].iloc[-1]
        duration_seconds = (end_time - start_time).total_seconds()
        observations = len(period)

        method_period_count += 1
        method_period_durations.append(duration_seconds)

        period_rows.append(
            {
                "method": method,
                "period_id": int(period_id),
                "start_timestamp": start_time,
                "end_timestamp": end_time,
                "duration_seconds": duration_seconds,
                "duration_minutes": duration_seconds / 60,
                "false_positive_observations": observations,
                "inferred_cadence_seconds": expected_cadence_seconds,
                "period_gap_limit_seconds": period_gap_limit_seconds,
            }
        )

    summary_rows.append(
        {
            "method": method,
            "validation_observations": len(method_df),
            "normal_observations": normal_count,
            "false_positive_observations": false_positive_count,
            "false_positive_rate": false_positive_rate,
            "false_alarm_periods": method_period_count,
            "mean_period_duration_seconds": (
                float(np.mean(method_period_durations))
                if method_period_durations
                else np.nan
            ),
            "median_period_duration_seconds": (
                float(np.median(method_period_durations))
                if method_period_durations
                else np.nan
            ),
            "longest_period_duration_seconds": (
                float(np.max(method_period_durations))
                if method_period_durations
                else np.nan
            ),
            "inferred_cadence_seconds": expected_cadence_seconds,
        }
    )

false_alarm_summary = (
    pd.DataFrame(summary_rows)
    .set_index("method")
)

false_alarm_periods = pd.DataFrame(
    period_rows,
    columns=[
        "method",
        "period_id",
        "start_timestamp",
        "end_timestamp",
        "duration_seconds",
        "duration_minutes",
        "false_positive_observations",
        "inferred_cadence_seconds",
        "period_gap_limit_seconds",
    ],
)

# 4. Save the results

summary_path = output_dir / "validation_false_alarm_summary.csv"
periods_path = output_dir / "validation_false_alarm_periods.csv"

false_alarm_summary.to_csv(summary_path, index=True)
false_alarm_periods.to_csv(periods_path, index=False)

# 5. Report and validate the results

display_columns = [
    "normal_observations",
    "false_positive_observations",
    "false_positive_rate",
    "false_alarm_periods",
    "mean_period_duration_seconds",
    "median_period_duration_seconds",
    "longest_period_duration_seconds",
]

print("\nVALIDATION FALSE-ALARM SUMMARY")
print("-" * 100)
print(
    false_alarm_summary[display_columns]
    .sort_values("false_positive_rate", ascending=False)
    .to_string(
        formatters={
            "false_positive_rate": lambda x: f"{x:.4%}",
            "mean_period_duration_seconds": lambda x: f"{x:.2f}",
            "median_period_duration_seconds": lambda x: f"{x:.2f}",
            "longest_period_duration_seconds": lambda x: f"{x:.2f}",
        }
    )
)

print("\nSaved files:")
# print(summary_path)
# print(periods_path)

assert len(false_alarm_summary) == len(METHODS)
assert (
    false_alarm_summary["false_positive_observations"]
    <= false_alarm_summary["normal_observations"]
).all()

assert (
    false_alarm_summary["false_alarm_periods"]
    == false_alarm_periods.groupby("method").size().reindex(
        METHODS, fill_value=0
    )
).all()

print("\nPASS: all nine methods were audited and the period counts reconcile.")

VALIDATION TIMESTAMP AUDIT
------------------------------------------------------------
Unique timestamps: 79,031
First timestamp:   2016-01-01 15:51:49
Last timestamp:    2016-01-02 14:59:59
Most frequent positive interval: 1 seconds
Share of intervals matching the most frequent interval: 99.99%

Most frequent observed intervals (seconds):
timestamp
1.0       79022
102.0         1
482.0         1
541.0         1
470.0         1
282.0         1
402.0         1
298.0         1
1691.0        1

VALIDATION FALSE-ALARM SUMMARY
----------------------------------------------------------------------------------------------------
         normal_observations  false_positive_observations false_positive_rate  false_alarm_periods mean_period_duration_seconds median_period_duration_seconds longest_period_duration_seconds
method                                                                                                                                                                             

In [26]:
# Purpose:
# Combine normal-validation false-alarm results with scenario-level
# detection results for a single, auditable comparison.

import pandas as pd

# Load the saved scenario summary to avoid relying on an assumed
# in-memory variable name or potentially stale notebook state.
scenario_summary_path = (
    Path(PROJECT_ROOT)
    / "nb09_artifacts"
    / "scenario_detection_summary.csv"
)

if not scenario_summary_path.exists():
    raise FileNotFoundError(
        f"Scenario summary not found: {scenario_summary_path}"
    )

scenario_summary_file = pd.read_csv(scenario_summary_path)

# Identify the method column without assuming its capitalization.
method_column = next(
    (
        column
        for column in scenario_summary_file.columns
        if column.strip().lower() in {"method", "unnamed: 0"}
    ),
    None,
)

if method_column is None:
    # The method name may have been saved as the first index column.
    method_column = scenario_summary_file.columns[0]

scenario_summary_file = scenario_summary_file.rename(
    columns={method_column: "method"}
)

scenario_summary_file["method"] = (
    scenario_summary_file["method"].astype(str).str.lower().str.strip()
)

if scenario_summary_file["method"].duplicated().any():
    raise ValueError(
        "Scenario summary contains duplicate method rows."
    )

required_scenario_columns = {
    "scenarios",
    "scenarios_detected",
    "scenarios_missed",
    "mean_scenario_coverage",
    "median_detection_delay_minutes",
    "scenario_detection_rate",
}

missing_scenario_columns = required_scenario_columns.difference(
    scenario_summary_file.columns
)

if missing_scenario_columns:
    raise KeyError(
        "Scenario summary is missing required columns: "
        f"{sorted(missing_scenario_columns)}. "
        f"Available columns: {list(scenario_summary_file.columns)}"
    )

# Prepare validation false-alarm summary.
false_alarm_table = false_alarm_summary.reset_index().copy()
false_alarm_table["method"] = (
    false_alarm_table["method"].astype(str).str.lower().str.strip()
)

# Keep only columns needed for the joint comparison.
false_alarm_columns = [
    "method",
    "normal_observations",
    "false_positive_observations",
    "false_positive_rate",
    "false_alarm_periods",
    "median_period_duration_seconds",
    "longest_period_duration_seconds",
]

scenario_columns = [
    "method",
    "scenarios",
    "scenarios_detected",
    "scenarios_missed",
    "mean_scenario_coverage",
    "median_detection_delay_minutes",
    "scenario_detection_rate",
]

comparison = false_alarm_table[false_alarm_columns].merge(
    scenario_summary_file[scenario_columns],
    on="method",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if not comparison["_merge"].eq("both").all():
    raise ValueError(
        "The method sets differ between false-alarm and scenario results:\n"
        + comparison.loc[
            ~comparison["_merge"].eq("both"),
            ["method", "_merge"],
        ].to_string(index=False)
    )

comparison = comparison.drop(columns="_merge")

if len(comparison) != len(METHODS):
    raise AssertionError(
        f"Expected {len(METHODS)} methods, found {len(comparison)}."
    )

# Sort by scenario detection rate, then lower false-positive rate.
comparison = comparison.sort_values(
    ["scenario_detection_rate", "false_positive_rate"],
    ascending=[False, True],
).reset_index(drop=True)

comparison_path = (
    Path(PROJECT_ROOT)
    / "nb09_artifacts"
    / "validation_false_alarms_vs_scenario_detection.csv"
)

comparison.to_csv(comparison_path, index=False)

display_columns = [
    "method",
    "false_positive_rate",
    "false_alarm_periods",
    "scenarios_detected",
    "scenarios_missed",
    "scenario_detection_rate",
    "mean_scenario_coverage",
    "median_detection_delay_minutes",
]

print("VALIDATION FALSE ALARMS VS SCENARIO DETECTION")
print("-" * 110)

print(
    comparison[display_columns].to_string(
        index=False,
        formatters={
            "false_positive_rate": lambda x: f"{x:.2%}",
            "scenario_detection_rate": lambda x: f"{x:.2%}",
            "mean_scenario_coverage": lambda x: f"{x:.2%}",
            "median_detection_delay_minutes": (
                lambda x: "N/A" if pd.isna(x) else f"{x:.3f}"
            ),
        },
    )
)

# print(f"\nSaved comparison: {comparison_path}")
print("\nPASS: all nine methods have matching validation and scenario summaries.")

VALIDATION FALSE ALARMS VS SCENARIO DETECTION
--------------------------------------------------------------------------------------------------------------
 method false_positive_rate  false_alarm_periods  scenarios_detected  scenarios_missed scenario_detection_rate mean_scenario_coverage median_detection_delay_minutes
rolling              41.34%                 6646                  36                 0                 100.00%                 40.99%                          0.000
   rule              92.74%                  232                  36                 0                 100.00%                 74.63%                          0.000
 simple              13.69%                  232                  33                 3                  91.67%                 56.97%                          0.033
   lstm               2.86%                  180                  30                 6                  83.33%                 42.92%                          0.025
    tcn           

In [27]:

# Purpose: Summarize validation false positives and verify counts
# against the existing point-level error inventory.

fp_rows = []

for method in METHODS:
    predictions = np.asarray(PRED[(method, "validation")]).reshape(-1)

    if not np.isin(predictions, [0, 1]).all():
        raise ValueError(
            f"{method}: validation predictions must contain only 0 and 1."
        )

    total_validation = len(predictions)
    false_positives = int((predictions == 1).sum())

    false_positive_rate = (
        false_positives / total_validation
        if total_validation
        else np.nan
    )

    fp_rows.append({
        "Method": method,
        "Total_Val": total_validation,
        "FP": false_positives,
        "FPR": false_positive_rate,
    })

fp_summary = pd.DataFrame(fp_rows).set_index("Method")

# Verify that the error inventory contains the expected methods and FP counts.
if "FP" not in error_inventory.columns:
    raise KeyError("error_inventory must contain an 'FP' column.")

if set(fp_summary.index) != set(error_inventory.index):
    raise ValueError(
        "Method names do not match between fp_summary and error_inventory."
    )

expected_fp = error_inventory["FP"].reindex(fp_summary.index)

if expected_fp.isna().any():
    raise ValueError("Missing FP counts in error_inventory.")

assert (
    fp_summary["FP"].astype(int)
    == expected_fp.astype(int)
).all(), "Validation FP counts do not match error_inventory."

# Save the basic summary.
fp_summary.to_csv(NB09_OUT_DIR / "fp_summary.csv")

print(fp_summary.round(4).to_string())
print("\nPASS: validation FP counts match error_inventory.")


         Total_Val     FP     FPR
Method                           
rule         79031  73295  0.9274
rolling      79031  32669  0.4134
simple       79031  10823  0.1369
if           79031   1155  0.0146
ocsvm        79031    650  0.0082
lof          79031   2147  0.0272
lstm         79031   2259  0.0286
tcn          79031   2528  0.0320
usad         79031   1261  0.0160

PASS: validation FP counts match error_inventory.


In [30]:
import matplotlib.pyplot as plt


# Purpose: Plot validation false-positive timestamps for all nine methods.

val_ts_series = pd.Series(pd.to_datetime(val_ts_arr, errors="coerce"))

if val_ts_series.isna().any():
    raise ValueError("Validation timestamp array contains invalid timestamps.")

if not val_ts_series.is_monotonic_increasing:
    raise ValueError("Validation timestamps are not in chronological order.")

fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)

for ax, method in zip(axes.ravel(), METHODS):
    predictions = np.asarray(
        PRED[(method, "validation")]
    ).reshape(-1)

    if len(predictions) != len(val_ts_series):
        raise ValueError(
            f"{method}: {len(predictions)} predictions but "
            f"{len(val_ts_series)} validation timestamps."
        )

    fp_mask = predictions == 1
    fp_ts = val_ts_series.loc[fp_mask]

    ax.scatter(
        fp_ts,
        np.zeros(len(fp_ts)),
        s=1,
        alpha=0.5,
        color="tab:red",
    )
    ax.set_title(f"{method} (n_fp={int(fp_mask.sum())})")
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)

fig.suptitle("Validation-period false positives per method")
fig.tight_layout()
fig.savefig(NB09_OUT_DIR / "fp_timeline.png", dpi=120)
plt.close(fig)

assert (NB09_OUT_DIR / "fp_timeline.png").exists()
print("PASS: false-positive timeline saved with aligned timestamps.")


PASS: false-positive timeline saved with aligned timestamps.


In [32]:

# Purpose: Identify sensors with the largest mean absolute training-standardized
# values among false-positive validation observations.

train_block = eligible_normal.iloc[:TRAIN_N]

# Verify sensor columns and numeric data.
missing_sensor_columns = set(SENSOR_COLS).difference(eligible_normal.columns)
if missing_sensor_columns:
    raise KeyError(
        f"Sensor columns missing from eligible_normal: "
        f"{sorted(missing_sensor_columns)}"
    )

if len(SENSOR_COLS) < 5:
    raise ValueError("At least five sensor columns are required for top-five attribution.")

train_sensors = train_block[SENSOR_COLS].apply(
    pd.to_numeric, errors="coerce"
)

if train_sensors.isna().any().any():
    raise ValueError(
        "Training sensor data contains missing or non-numeric values. "
        "Resolve this before calculating attribution."
    )

sens_mean = train_sensors.mean()
sens_std = train_sensors.std()
valid_sensors = sens_std.index[
    sens_std.notna() & sens_std.gt(0)
]

if len(valid_sensors) < 5:
    raise ValueError(
        "Fewer than five sensors have a valid, non-zero training standard deviation."
    )

# Map validation sequence endpoints to the original eligible-normal rows.
val_end_global_idx = val_start + np.asarray(validation_end_indices)

if len(val_end_global_idx) != len(val_ts_series):
    raise ValueError(
        "Validation endpoint indices and validation timestamps have different lengths."
    )

if len(val_end_global_idx) != len(PRED[(METHODS[0], "validation")]):
    raise ValueError(
        "Validation endpoint indices and prediction arrays are not aligned."
    )

if (
    (val_end_global_idx < 0).any()
    or (val_end_global_idx >= len(eligible_normal)).any()
):
    raise IndexError("Validation endpoint indices fall outside eligible_normal.")

val_endpoint_raw = (
    eligible_normal.iloc[val_end_global_idx][SENSOR_COLS]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True)
)

if val_endpoint_raw.isna().any().any():
    raise ValueError(
        "Validation endpoint sensor values contain missing or non-numeric data."
    )

# Use training-only means and standard deviations.
val_z = (
    (val_endpoint_raw[valid_sensors] - sens_mean[valid_sensors])
    / sens_std[valid_sensors]
).abs()

attribution_rows = []

for method in METHODS:
    fp_mask = np.asarray(
        PRED[(method, "validation")]
    ).reshape(-1) == 1

    if len(fp_mask) != len(val_z):
        raise ValueError(
            f"{method}: false-positive mask and sensor rows are not aligned."
        )

    row = {"Method": method}

    if not fp_mask.any():
        for rank in range(1, 6):
            row[f"top{rank}"] = None
            row[f"top{rank}_absz"] = np.nan
    else:
        mean_abs_z = (
            val_z.loc[fp_mask]
            .mean(axis=0)
            .sort_values(ascending=False)
        )

        for rank, (sensor, abs_z) in enumerate(
            mean_abs_z.head(5).items(), start=1
        ):
            row[f"top{rank}"] = sensor
            row[f"top{rank}_absz"] = float(abs_z)

    attribution_rows.append(row)

fp_sensor_attribution = (
    pd.DataFrame(attribution_rows)
    .set_index("Method")
)

fp_sensor_attribution.to_csv(
    NB09_OUT_DIR / "fp_sensor_attribution.csv"
)

print(
    fp_sensor_attribution[
        ["top1", "top1_absz", "top2", "top2_absz", "top3", "top3_absz"]
    ].round(3).to_string()
)

assert len(fp_sensor_attribution) == len(METHODS)
assert fp_sensor_attribution.shape[1] == 10
assert set(fp_sensor_attribution.index) == set(METHODS)

print("\nPASS: sensor attribution calculated with training-only standardization.")


           top1  top1_absz    top2  top2_absz    top3  top3_absz
Method                                                          
rule     AIT203      3.494  AIT202      2.750  AIT201      2.744
rolling  AIT203      3.535  AIT202      2.774  AIT201      2.734
simple   AIT203      3.846  AIT202      2.999  AIT201      2.736
if         P602      8.299  FIT601      8.040   MV301      7.820
ocsvm    FIT504      6.685  AIT203      4.258  FIT401      4.067
lof      AIT203      3.349  AIT502      2.758  AIT201      2.686
lstm     AIT203      3.687  AIT502      3.061  AIT202      2.900
tcn      AIT203      4.129  AIT202      3.142  AIT502      3.057
usad     FIT601      4.337  AIT203      3.880   MV301      3.637

PASS: sensor attribution calculated with training-only standardization.


In [33]:

# Purpose: Summarize attack-partition false negatives and verify
# counts against the existing point-level error inventory.

fn_rows = []

for method in METHODS:
    predictions = np.asarray(
        PRED[(method, "attack")]
    ).reshape(-1)

    if not np.isin(predictions, [0, 1]).all():
        raise ValueError(
            f"{method}: attack predictions must contain only 0 and 1."
        )

    if len(predictions) != len(ATK_TIMESTAMPS):
        raise ValueError(
            f"{method}: prediction count does not match attack timestamps."
        )

    fn = int((predictions == 0).sum())
    total_attack = len(predictions)

    fnr = fn / total_attack if total_attack else np.nan

    fn_rows.append({
        "Method": method,
        "Total_Atk": total_attack,
        "FN": fn,
        "FNR": fnr,
    })

fn_summary = pd.DataFrame(fn_rows).set_index("Method")

if "FN" not in error_inventory.columns:
    raise KeyError("error_inventory must contain an 'FN' column.")

if set(fn_summary.index) != set(error_inventory.index):
    raise ValueError(
        "Method names do not match between fn_summary and error_inventory."
    )

expected_fn = error_inventory["FN"].reindex(fn_summary.index)

if expected_fn.isna().any():
    raise ValueError("Missing FN counts in error_inventory.")

assert (
    fn_summary["FN"].astype(int)
    == expected_fn.astype(int)
).all(), "Attack FN counts do not match error_inventory."

fn_summary.to_csv(NB09_OUT_DIR / "fn_summary.csv")

print(fn_summary.round(4).to_string())
print("\nPASS: attack FN counts match error_inventory.")


         Total_Atk     FN     FNR
Method                           
rule         54592   7083  0.1297
rolling      54592  34605  0.6339
simple       54592   9659  0.1769
if           54592  21356  0.3912
ocsvm        54592  20063  0.3675
lof          54592  51237  0.9385
lstm         54592  13546  0.2481
tcn          54592  11247  0.2060
usad         54592  19457  0.3564

PASS: attack FN counts match error_inventory.


In [35]:
# Purpose: Plot false-negative timestamps across the attack partition for all nine methods.

atk_ts_series = pd.Series(pd.to_datetime(ATK_TIMESTAMPS, errors="coerce"))

if atk_ts_series.isna().any():
    raise ValueError("ATK_TIMESTAMPS contains invalid or missing timestamps.")

fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)

for ax, m in zip(axes.ravel(), METHODS):
    pa = np.asarray(PRED[(m, "attack")])

    if pa.ndim != 1 or len(pa) != len(atk_ts_series):
        raise ValueError(
            f"{m}: attack predictions are not aligned with ATK_TIMESTAMPS."
        )

    if not np.isin(pa, [0, 1]).all():
        raise ValueError(f"{m}: attack predictions contain values other than 0 and 1.")

    fn_ts = atk_ts_series.iloc[np.flatnonzero(pa == 0)]

    ax.scatter(
        fn_ts,
        np.zeros(len(fn_ts)),
        s=1,
        alpha=0.5,
        color="tab:orange",
    )
    ax.set_title(f"{m} (n_fn={len(fn_ts):,})")
    ax.set_yticks([])
    ax.grid(True, alpha=0.3)

fig.suptitle("Attack-period false negatives per method")
fig.tight_layout()
fig.savefig(NB09_OUT_DIR / "fn_timeline.png", dpi=120)
plt.close(fig)

assert (NB09_OUT_DIR / "fn_timeline.png").exists()

print("PASS: False-negative timeline saved.")
# print(f"Output: {NB09_OUT_DIR / 'fn_timeline.png'}")
print("False-negative counts:")
for m in METHODS:
    pa = np.asarray(PRED[(m, "attack")])
    print(f"{m:8s} {int((pa == 0).sum()):,}")

PASS: False-negative timeline saved.
False-negative counts:
rule     7,083
rolling  34,605
simple   9,659
if       21,356
ocsvm    20,063
lof      51,237
lstm     13,546
tcn      11,247
usad     19,457


**Finding:** False-negative counts vary substantially across methods, with LOF missing the most attack-partition observations (51,237) and rule-based detection missing the fewest (7,083). The timeline shows when missed observations occur but does not establish their causes or indicate how many complete attack scenarios were missed.

In [37]:
# Purpose: Inspect scenario IDs and available eligibility fields without modifying data.

print("attack_scenarios shape:", attack_scenarios.shape)
print("Columns:", attack_scenarios.columns.tolist())
print("\nScenario IDs:")
print(sorted(attack_scenarios["attack_id"].dropna().unique().tolist()))

print("\nScenario table preview:")
display(attack_scenarios.head())

attack_scenarios shape: (41, 9)
Columns: ['attack_id', 'start_time', 'end_time', 'attack_points', 'start_state', 'attack', 'actual_change', 'description', 'unexpected_outcome']

Scenario IDs:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]

Scenario table preview:


,attack_id,start_time,end_time,attack_points,start_state,attack,actual_change,description,unexpected_outcome
0,1,2015-12-28 10:29:14,2015-12-28 10:44:53,MV-101,MV-101 is closed,Open MV-101,Yes,Tank overflow,
1,2,2015-12-28 10:51:08,2015-12-28 10:58:30,P-102,P-101 is on where as P-102 is off,Turn on P-102,Yes,Pipe bursts,
2,3,2015-12-28 11:22:00,2015-12-28 11:28:22,LIT-101,Water level between L and H,Increase by 1 mm every second,No,Tank Underflow; Damage P-101,
3,4,2015-12-28 11:47:39,2015-12-28 11:54:08,MV-504,MV-504 is closed,Open MV-504,Yes,Halt RO shut down sequence; Reduce life of RO,No impact
4,5,2015-12-28 11:58:20,NaT,No Physical Impact Attack,,,,,


In [38]:
# Purpose: Verify scenario interval eligibility against the established 36-scenario analysis.

scenario_check = attack_scenarios.copy()

scenario_check["start_time"] = pd.to_datetime(
    scenario_check["start_time"], errors="coerce"
)
scenario_check["end_time"] = pd.to_datetime(
    scenario_check["end_time"], errors="coerce"
)

scenario_check["valid_interval"] = (
    scenario_check["start_time"].notna()
    & scenario_check["end_time"].notna()
    & (scenario_check["end_time"] > scenario_check["start_time"])
)

print("Total workbook records:", len(scenario_check))
print("Records with valid intervals:", int(scenario_check["valid_interval"].sum()))
print("Records without valid intervals:")
display(
    scenario_check.loc[
        ~scenario_check["valid_interval"],
        ["attack_id", "start_time", "end_time", "description"],
    ]
)

Total workbook records: 41
Records with valid intervals: 36
Records without valid intervals:


,attack_id,start_time,end_time,description
4,5,2015-12-28 11:58:20,NaT,
8,9,2015-12-28 14:15:00,NaT,
11,12,2015-12-29 11:10:40,NaT,
14,15,2015-12-29 11:52:01,NaT,
17,18,2015-12-29 18:08:55,NaT,


In [39]:
# Purpose: Calculate per-method detection and coverage for the 36 complete scenarios.

if not HAVE_SCENARIOS:
    print("BLOCKED: HAVE_SCENARIOS=False; skipping scenario detection matrix.")
    scenario_detection = None

else:
    scenario_table = attack_scenarios.copy()
    scenario_table["start_time"] = pd.to_datetime(
        scenario_table["start_time"], errors="coerce"
    )
    scenario_table["end_time"] = pd.to_datetime(
        scenario_table["end_time"], errors="coerce"
    )

    valid_interval = (
        scenario_table["start_time"].notna()
        & scenario_table["end_time"].notna()
        & (scenario_table["end_time"] > scenario_table["start_time"])
    )
    eligible_scenarios = scenario_table.loc[valid_interval].copy()
    scenario_ids = sorted(eligible_scenarios["attack_id"].astype(int).tolist())

    if len(scenario_ids) != 36 or len(set(scenario_ids)) != 36:
        raise ValueError(
            f"Expected 36 unique complete scenarios; found {len(scenario_ids)}."
        )

    if len(atk_seq_df) != len(ATK_TIMESTAMPS):
        raise ValueError("Scenario rows are not aligned with attack timestamps.")

    rows = []

    for m in METHODS:
        pa = np.asarray(PRED[(m, "attack")])

        if pa.ndim != 1 or len(pa) != len(atk_seq_df):
            raise ValueError(
                f"{m}: attack predictions are not aligned with scenario rows."
            )

        if not np.isin(pa, [0, 1]).all():
            raise ValueError(f"{m}: predictions must contain only 0 and 1.")

        for sid in scenario_ids:
            mask = (atk_seq_df["scenario_id"] == sid).to_numpy()
            n_seqs = int(mask.sum())

            if n_seqs == 0:
                raise ValueError(
                    f"Scenario {sid} has no matched attack sequence endpoints."
                )

            n_flagged = int(pa[mask].sum())

            rows.append({
                "Method": m,
                "scenario_id": sid,
                "n_seqs": n_seqs,
                "n_flagged": n_flagged,
                "detected": int(n_flagged > 0),
                "coverage": n_flagged / n_seqs,
            })

    scenario_detection = pd.DataFrame(rows)

    scenario_detection.to_csv(
        NB09_OUT_DIR / "scenario_detection.csv",
        index=False,
    )

    pivot_detected = (
        scenario_detection
        .pivot(index="scenario_id", columns="Method", values="detected")
        .reindex(columns=METHODS)
    )
    pivot_detected.to_csv(
        NB09_OUT_DIR / "scenario_detection_pivot.csv"
    )

    assert len(scenario_detection) == 36 * len(METHODS)
    assert scenario_detection["scenario_id"].nunique() == 36
    assert set(scenario_detection["Method"].unique()) == set(METHODS)
    assert not scenario_detection.duplicated(
        ["Method", "scenario_id"]
    ).any()

    print(f"Complete scenarios: {len(scenario_ids)}")
    print(f"Scenario detection rows: {len(scenario_detection)}")
    print("\nScenario detection matrix (1 = at least one endpoint flagged):")
    print(pivot_detected.astype(int).to_string())
    print("\nPASS: Scenario detection and coverage tables saved.")

Complete scenarios: 36
Scenario detection rows: 324

Scenario detection matrix (1 = at least one endpoint flagged):
Method       rule  rolling  simple  if  ocsvm  lof  lstm  tcn  usad
scenario_id                                                        
1               1        1       1   0      0    1     1    0     0
2               1        1       1   1      0    1     1    1     1
3               1        1       1   0      0    0     0    1     0
4               1        1       1   0      0    0     1    0     1
6               1        1       1   0      0    0     1    1     0
7               1        1       1   0      0    0     1    1     0
8               1        1       1   1      0    0     1    1     1
10              1        1       1   0      1    1     1    1     1
11              1        1       1   1      1    1     1    1     1
13              1        1       0   0      0    0     0    0     0
14              1        1       1   0      0    0     0    0     0


In [40]:
# Purpose: Per (method, scenario_id) detection and coverage; save detection + pivot.
if not HAVE_SCENARIOS:
    print("BLOCKED: HAVE_SCENARIOS=False; skipping scenario detection matrix.")
    scenario_detection = None
else:
    rows = []
    for m in METHODS:
        pa = PRED[(m, "attack")]
        for sid in sorted(attack_scenarios["attack_id"].unique()):
            mask = (atk_seq_df["scenario_id"] == sid).values
            n_seqs = int(mask.sum())
            if n_seqs == 0:
                continue
            n_flagged = int(pa[mask].sum())
            rows.append({"Method": m, "scenario_id": sid,
                         "n_seqs": n_seqs, "n_flagged": n_flagged,
                         "detected": int(n_flagged > 0),
                         "coverage": n_flagged / n_seqs})
    scenario_detection = pd.DataFrame(rows)
    scenario_detection.to_csv(NB09_OUT_DIR / "scenario_detection.csv", index=False)

    pivot_detected = (scenario_detection
                      .pivot(index="scenario_id", columns="Method", values="detected")[METHODS])
    pivot_detected.to_csv(NB09_OUT_DIR / "scenario_detection_pivot.csv")

    assert scenario_detection["scenario_id"].nunique() == 36
    assert set(scenario_detection["Method"].unique()) == set(METHODS)
    print(f"scenario_detection rows = {len(scenario_detection)}")
    print(pivot_detected.astype(int).to_string())

scenario_detection rows = 324
Method       rule  rolling  simple  if  ocsvm  lof  lstm  tcn  usad
scenario_id                                                        
1               1        1       1   0      0    1     1    0     0
2               1        1       1   1      0    1     1    1     1
3               1        1       1   0      0    0     0    1     0
4               1        1       1   0      0    0     1    0     1
6               1        1       1   0      0    0     1    1     0
7               1        1       1   0      0    0     1    1     0
8               1        1       1   1      0    0     1    1     1
10              1        1       1   0      1    1     1    1     1
11              1        1       1   1      1    1     1    1     1
13              1        1       0   0      0    0     0    0     0
14              1        1       1   0      0    0     0    0     0
16              1        1       1   0      0    0     1    0     0
17              1 